# 🧬 Diccionario Humano de Citoquinas — Práctica de Inmunología (v3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garridolab/cytokines-practical/blob/main/CytokineDictionary_Practical_v3_es.ipynb)

**Basada en:** Oesinghaus *et al.* (2025), *A single-cell cytokine dictionary of human peripheral blood*, bioRxiv
([doi:10.64898/2025.12.12.693897](https://doi.org/10.64898/2025.12.12.693897))

En el artículo, ~9.7 millones de células sanguíneas humanas (PBMC) de **12 donantes** se estimularon con **90 citoquinas** (o con un control de PBS)
y se analizaron mediante RNA-seq de célula única. Aquí trabajarás con una versión **pseudobulk** de los datos: para cada combinación
*donante × citoquina × tipo celular*, los conteos de todas las células se **sumaron** en un único perfil de expresión tipo bulk.
Usamos **cuatro tipos celulares**: **linfocitos T CD4, linfocitos T CD8, células NK (CD56low) y monocitos CD14**.
(El archivo que usamos contiene **87 de las citoquinas** más el control de PBS, para los 12 donantes.)

> 💬 Los nombres de las columnas, de las variables y los valores de los datos (`donor`, `cytokine`, `treated`…) se mantienen en inglés, igual que en el código y en las diapositivas.

### El plan (2 h 30 min)

| Parte | Tema | Formato | Tiempo |
|:--:|---|---|:--:|
| **1** | **Introducción**: Python, Jupyter y Colab, RNA-seq, célula única, pseudobulk, el conjunto de datos, AnnData | diapositivas + 5 min de preparación aquí | 25 min |
| **2** | **Manipulación y normalización de los datos** | este notebook | 40 min |
| ☕ | *Descanso* | | 10 min |
| **3** | **Análisis de Componentes Principales (PCA)**: un análisis para dominarlos a todos | diapositivas + notebook | 35 min |
| **4** | **Expresión diferencial**: citoquina vs. control | diapositivas + notebook | 35 min |
| | Cierre | | 5 min |

### Cómo leer este notebook

| Símbolo | Significado |
|:--:|---|
| ⏱ | Tiempo previsto de la sección |
| 🔧 | **Te toca / CAMBIA ESTO**: una tarea para ti. Las celdas de código que empiezan con `# 🔧 CAMBIA ESTO` contienen parámetros que puedes modificar: cámbialos, vuelve a ejecutar y observa qué pasa |
| ❓ | **Pregunta**: piénsala *antes* de abrir la respuesta |
| 💡 | Consejo / algo que conviene recordar |
| ⚠️ | Una trampa frecuente |
| 🎁 | Extra: solo si te sobra tiempo (o en casa) |

> **Ejecutar el notebook:** ejecuta las celdas **de arriba abajo** con **Shift + Enter**.
> Si te pierdes, hay una **celda de rescate** al inicio de la Parte 3: reconstruye en un solo paso los datos de la Parte 2.

---
# Parte 1 · Introducción

Las ideas (Python, Jupyter, Colab, RNA-seq, célula única, pseudobulk, el conjunto de datos, AnnData) se explican en las **diapositivas**.
Aquí solo necesitas **prepararte**: son unos 5 minutos. ⏱ *5 min*

## 1.1 · Jupyter en 60 segundos

* Un notebook es una lista de **celdas**. Algunas contienen **texto** (Markdown, como esta) y otras contienen **código** (Python).
* Para **ejecutar** una celda de código: haz clic en ella y pulsa **Shift + Enter** (o el botón ▶). El resultado aparece justo debajo.
* Las celdas comparten su **memoria**: una variable creada en una celda existe en todas las siguientes. Ejecuta las celdas **en orden**.
* ¿Algo ha ido mal? **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar las celdas desde el principio.

Pruébalo:

In [1]:
# Esta es una celda de código: haz clic en ella y pulsa Shift+Enter.
# Crea una variable llamada `message`, que ahora vive en la memoria del notebook.
# 🔧 CAMBIA ESTO: escribe tu propio nombre
message = "¡Hola, inmunólogos!"

In [2]:
print(message)          # esta celda usa la variable creada en la celda anterior
6 * 7                   # el valor de la ÚLTIMA línea de una celda se muestra automáticamente

¡Hola, inmunólogos!


42

> 🔧 **Te toca:** cambia `message` por tu nombre, ejecuta la primera celda y después la segunda.
> Luego haz **Entorno de ejecución → Reiniciar sesión** y ejecuta *solo la segunda celda*: obtendrás un `NameError`. **El orden importa**.

## 1.2 · Instalar e importar los paquetes

Colab ya incluye casi todo lo que necesitamos. La celda siguiente instala solo lo que falta (`anndata`, …).

In [ ]:
# ── Instalar las dependencias que falten ─────────────────────────────────────────
import importlib.util, shutil, subprocess, sys

REQUIRED = {                       # nombre de importación -> nombre del paquete en PyPI
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "statsmodels": "statsmodels", "anndata": "anndata",
    "gdown": "gdown", "adjustText": "adjustText", "pydeseq2": "pydeseq2",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f"📦 Instalando: {', '.join(missing)} …")
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    subprocess.check_call(cmd)
    importlib.invalidate_caches()
    print("✅ Hecho")
else:
    print("✅ Todo está ya instalado")

In [ ]:
#@title Importaciones, colores y ajustes (ejecutar una vez, no hace falta leer) { display-mode: "form" }
import os, warnings
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
rng = np.random.default_rng(42)                # un único generador aleatorio para todo el notebook

# ── Estilo de los gráficos ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 10.5, "axes.labelsize": 9.5,
    "legend.frameon": False, "legend.fontsize": 8,
})

# ── Los cuatro tipos celulares y los 12 donantes ────────────────────────────────────
# Los colores proceden de una paleta categórica comprobada para daltonismo; cada tipo celular tiene además su
# propia forma de marcador, así que la identidad nunca depende solo del color.
CELL_TYPES = ["CD4_T_cell", "CD8_T_cell", "NK_CD56low", "CD14_Mono"]
CT_LABEL   = {"CD4_T_cell": "linfocito T CD4", "CD8_T_cell": "linfocito T CD8", "NK_CD56low": "NK CD56low", "CD14_Mono": "monocito CD14"}
CT_SHORT   = {"CD4_T_cell": "T CD4", "CD8_T_cell": "T CD8", "NK_CD56low": "NK", "CD14_Mono": "Mono"}
CT_COLORS  = {"CD4_T_cell": "#2a78d6", "CD8_T_cell": "#008300", "NK_CD56low": "#eda100", "CD14_Mono": "#e87ba4"}
CT_MARKERS = {"CD4_T_cell": "o", "CD8_T_cell": "s", "NK_CD56low": "^", "CD14_Mono": "D"}
DONORS     = [f"Donor{i}" for i in range(1, 13)]

# ── Colores para la "dirección": azul (baja) ↔ gris ↔ rojo (sube) ───────────────────
UP_COLOR, DOWN_COLOR, NS_COLOR, PBS_COLOR = "#e34948", "#2a78d6", "#c3c2b7", "#8a8984"
DIVERGING  = LinearSegmentedColormap.from_list("div", [DOWN_COLOR, "#f0efec", UP_COLOR])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"])

print("✅ Importaciones correctas")

## 1.3 · Descargar y cargar los datos

Los datos son un único archivo `.h5ad` (≈ 500 MB, un objeto **AnnData** guardado en disco): los conteos pseudobulk de 4 tipos celulares,
restringidos a 18,120 genes codificantes de proteínas. En Colab la descarga suele tardar uno o dos minutos.

In [ ]:
# Archivo de Google Drive con los datos pseudobulk (compartido con "cualquiera con el enlace")
GDRIVE_FILE_ID = "1thFoQnNVVqgKvspg5b5k2fBgjk4bEqAn"
# /content existe en Colab; en local el archivo se guarda junto al notebook
DATA_PATH = "/content/pseudobulks_4ct_proteincoding.h5ad" if os.path.isdir("/content") else "pseudobulks_4ct_proteincoding.h5ad"

if not os.path.exists(DATA_PATH):
    import gdown
    print("⬇️  Descargando de Google Drive …")
    os.makedirs(os.path.dirname(os.path.abspath(DATA_PATH)), exist_ok=True)
    out = gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", DATA_PATH, quiet=False)
    if out is None or not os.path.exists(DATA_PATH):
        raise RuntimeError("La descarga ha fallado: inténtalo de nuevo en un minuto (cuota de Drive) o pide el archivo.")
print(f"✅ Archivo listo: {DATA_PATH}  ({os.path.getsize(DATA_PATH) / 1e9:.2f} GB)")

adata_raw = ad.read_h5ad(DATA_PATH)        # leer el archivo en memoria: un objeto AnnData
print("✅ Cargado en memoria")

Eso es todo para la Parte 1 en el notebook. 👉 **Vuelve a las diapositivas** para la introducción; cuando lleguemos a la diapositiva de *AnnData*, regresa aquí.

---
# Parte 2 · Manipulación y normalización de los datos
⏱ *40 min* — **Objetivo:** entender qué hay dentro de los datos y convertir los conteos crudos en números que podamos comparar.

| Paso | Pregunta | Tiempo |
|:--:|---|:--:|
| **2.1** | Preparar el objeto | 3 min |
| **2.2** | ¿Qué aspecto tienen los datos? De **1 número** a la **matriz completa** | 10 min |
| **2.3** | ¿Qué es un **conteo**? ¿Por qué las librerías tienen tamaños distintos? | 7 min |
| **2.4** | **Normalización**: qué, por qué, cómo | 10 min |
| **2.5** | ¿Por qué **filtramos** los genes poco expresados? | 5 min |
| **2.6** | **Mapas de calor** antes y después de normalizar | 5 min |

## 2.1 · Preparar el objeto

El archivo que hemos cargado contiene algunos restos del objeto original de célula única. Esta celda lo ordena
(explicamos el *porqué* en el extra 🎁 *"El rompecabezas del PBS"* al final; por ahora, simplemente ejecútala):

1. el **control PBS** se midió en 6 pocillos, y el archivo guarda cada pocillo *más* su total → nos quedamos solo con el **total**;
2. conservamos los **metadatos de las muestras** que necesitamos y damos a cada muestra un **nombre único** y legible: `donor|cell type|cytokine`;
3. convertimos la matriz en una tabla de números normal (densa);
4. añadimos dos columnas **a nivel de gen** propias: los conteos totales y en cuántas muestras se detecta el gen.

In [ ]:
# 1) Control PBS: conservar la fila total ya calculada y descartar los 6 pocillos individuales que se sumaron en ella
is_pbs_well = ((adata_raw.obs["cytokine"] == "PBS") & adata_raw.obs["bc1_well"].notna()).values
adata = adata_raw[~is_pbs_well].copy()
if "psbulk_props" in adata.layers:
    del adata.layers["psbulk_props"]                 # una capa que no usamos

# 2) Metadatos de las muestras (obs): conservar las columnas útiles, fijar el orden de las categorías y dar nombres únicos
obs = adata.obs[["donor", "cytokine", "treatment", "cell_type", "psbulk_n_cells"]].copy()
cytokines = ["PBS"] + sorted(c for c in obs["cytokine"].astype(str).unique() if c != "PBS")
obs["donor"]     = pd.Categorical(obs["donor"].astype(str), categories=DONORS)
obs["cytokine"]  = pd.Categorical(obs["cytokine"].astype(str), categories=cytokines)
obs["cell_type"] = pd.Categorical(obs["cell_type"].astype(str), categories=CELL_TYPES)
obs["treatment"] = pd.Categorical(np.where(obs["cytokine"] == "PBS", "control", "treated"), categories=["control", "treated"])
obs.index = (obs["donor"].astype(str) + "|" + obs["cell_type"].astype(str) + "|" + obs["cytokine"].astype(str)).values
assert obs.index.is_unique
adata.obs = obs

# 3) La matriz X: conteos crudos como tabla densa de números (el 80 % de las entradas no son cero)
X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
adata.X = X.astype(np.float32)

# 4) Metadatos de los genes (var): empezar de cero y añadir dos columnas a nivel de gen calculadas a partir de los conteos
adata.var = pd.DataFrame(index=adata.var_names)
adata.var["total_counts"]       = adata.X.sum(axis=0)          # conteos del gen, sumados en todas las muestras
adata.var["n_samples_detected"] = (adata.X > 0).sum(axis=0)    # en cuántas muestras el gen tiene al menos 1 conteo

print("✅ adata está listo")

## 2.2 · Un número cada vez: de 1 × 1 a la matriz completa

Un objeto **AnnData** (lo explicaron las diapositivas) mantiene tres tablas **alineadas**: `X` (los números), `obs` (una fila por muestra) y `var` (una fila por gen).
Seleccionamos trozos con `adata[ muestras , genes ]` — **primero las muestras, después los genes** — usando *nombres*.

### La pieza más pequeña: 1 muestra × 1 gen

In [ ]:
# 🔧 CAMBIA ESTO: cualquier muestra ("Donor7|CD4_T_cell|IL-2") y cualquier gen ("CD3E", "IL7R", ...)
sample = "Donor1|CD14_Mono|IFN-gamma"     # donante | tipo celular | citoquina
gene   = "STAT1"

one = adata[[sample], [gene]]             # adata[ lista de muestras , lista de genes ]
print(one)

In [ ]:
print("X: la medida\n")
display(one.to_df())                      # una tabla de 1 × 1: esta muestra, este gen

print("obs: metadatos de la MUESTRA (una fila por muestra)\n")
display(one.obs)

print("var: metadatos del GEN (una fila por gen)\n")
display(one.var)

**¿Qué estamos viendo?**

* `X` es **un número**: el **conteo** del gen `STAT1` en esta muestra: cuántas moléculas de mRNA (UMI) se detectaron en *todos* los monocitos CD14 del Donor 1 estimulados con IFN-γ, sumados.
* `obs` describe la **muestra**: el donante, la citoquina, si es control o tratada, el tipo celular y **cuántas células se sumaron** (`psbulk_n_cells`).
* `var` describe el **gen**: aquí, sus conteos totales en todas las muestras y en cuántas muestras se detectó.

### Una muestra, cinco genes (1 × 5)

La misma muestra, ahora con cinco genes: tres **marcadores de tipo celular** (`LYZ`: monocitos, `CD3E`: linfocitos T, `NKG7`: células NK),
un **gen de respuesta a interferón** (`STAT1`) y un **gen constitutivo** (`ACTB`, que se espera siempre activo).

In [ ]:
# 🔧 CAMBIA ESTO: los cinco genes que quieras
genes5 = ["LYZ", "CD3E", "NKG7", "STAT1", "ACTB"]
five_genes = adata[[sample], genes5]
print(five_genes)                         # 1 muestra × 5 genes: AnnData siempre escribe  n_obs × n_vars  (muestras × genes)

display(five_genes.to_df())               # una fila (la muestra), cinco columnas (los genes)
display(five_genes.var)                   # cinco filas: una por gen

### Cinco muestras, cinco genes (5 × 5)

Ahora cinco **muestras** del Donor 1: el **control (PBS)** de cada uno de los cuatro tipos celulares, más los monocitos estimulados con IFN-γ.

In [ ]:
# 🔧 CAMBIA ESTO: otro donante ("Donor5") u otra citoquina en lugar de IFN-gamma
samples5 = [f"Donor1|{ct}|PBS" for ct in CELL_TYPES] + ["Donor1|CD14_Mono|IFN-gamma"]
five_by_five = adata[samples5, genes5]
print(five_by_five)

print("\nX: conteos crudos (filas = muestras, columnas = genes)")
display(five_by_five.to_df())
print("obs: las cinco muestras")
display(five_by_five.obs)

**❓ Preguntas**
1. Para `LYZ`, `CD3E` y `NKG7`: ¿qué muestra tiene el valor más alto? ¿Coincide con lo que sabes de estos marcadores?
2. `ACTB` debería estar "siempre activo", pero sus conteos difieren mucho entre filas. Mira `psbulk_n_cells`: ¿ves alguna relación?
3. `STAT1` es inducido por IFN-γ. Compara las dos filas de **monocitos** (PBS vs IFN-γ): ¿es mayor el conteo crudo de `STAT1` con IFN-γ? ¿Es lo que esperabas?

<details>
<summary>💡 Respuestas</summary>

1. `LYZ` → monocitos; `CD3E` → linfocitos T (la muestra de T CD4 tiene el valor más alto, la de T CD8 el segundo); `NKG7` → células NK, y después T CD8 (también citotóxicos).
2. Sí: más células sumadas → más conteos. Compara las dos filas de monocitos: 17,373 células vs 2,556 células → 260,422 vs 36,500 conteos de `ACTB`.
3. **No: es ~9× *menor* con IFN-γ** (73,853 vs 639,289), aunque IFN-γ induce `STAT1`. La muestra PBS es la suma de 6 pocillos y tiene 7× más células.
   **Los conteos crudos mezclan la biología con la cantidad de material**: ese es el objetivo de las dos secciones siguientes.

</details>

### Seleccionar con una condición (máscara booleana)

Los nombres sirven para unas pocas muestras. Para seleccionar *muchas*, construye una **máscara booleana** sobre `obs` (un valor Verdadero/Falso para cada muestra) y úsala como primer índice.
Aquí: todas las muestras de monocitos que son control o tratadas con IFN-γ (12 donantes × 2 = 24 muestras), para dos dianas del interferón-γ, `IDO1` y `CXCL10`:

In [ ]:
# 🔧 CAMBIA ESTO: el tipo celular, las citoquinas y los genes
mask = (adata.obs["cell_type"] == "CD14_Mono") & adata.obs["cytokine"].isin(["PBS", "IFN-gamma"])
sub  = adata[mask, ["IDO1", "CXCL10"]]
print(sub, "\n")

table = sub.to_df()
table["donor"]    = sub.obs["donor"].astype(str).values
table["cytokine"] = sub.obs["cytokine"].astype(str).values
display(table.pivot(index="donor", columns="cytokine", values=["IDO1", "CXCL10"]).loc[DONORS])    # 12 donantes × (2 genes × 2 condiciones)

Mira con atención: en varios donantes los **conteos de PBS son tan altos como los de IFN-γ, o más**, ¡y sin embargo IFN-γ es un potente inductor de ambos genes!
Las muestras de PBS tienen ~5× más células (son la suma de 6 pocillos), así que tienen más conteos de *todo*. Los conteos crudos no se pueden comparar entre muestras; lo arreglaremos en 2.3–2.4.

### Alejando la vista: la matriz completa

In [ ]:
print(adata)
print("\nForma de la matriz completa (muestras × genes):", adata.shape)
print(f"→ {adata.n_obs:,} muestras × {adata.n_vars:,} genes = {adata.n_obs * adata.n_vars:,} números")
print(f"→ el diseño: 4 tipos celulares × 88 condiciones (87 citoquinas + PBS) × 12 donantes = {4 * 88 * 12:,} muestras")
print(f"→ {np.mean(adata.X == 0):.1%} de las entradas son ceros")
print(f"→ {adata.X.nbytes / 1e6:.0f} MB en memoria\n")

display(pd.crosstab(adata.obs["cell_type"], adata.obs["treatment"]))

La tabla de 5 × 5 que imprimimos antes era solo la **esquina superior izquierda** de esta gran tabla. Todo lo que hagamos a continuación se aplica a la matriz *completa* de una vez.

> 🔧 **Te toca** (2 min)
> 1. ¿Cuántas muestras de `NK_CD56low` se trataron con `IL-2`? *(pista: construye una máscara con `&` y luego toma `.n_obs` del subconjunto)*
> 2. Obtén los conteos crudos de `CXCL9` en las 12 muestras de `CD14_Mono` tratadas con `IL-4`. ¿Qué donante tiene el valor más alto?

<details>
<summary>💡 Soluciones</summary>

```python
m = (adata.obs["cell_type"] == "NK_CD56low") & (adata.obs["cytokine"] == "IL-2")
adata[m].n_obs                                    # 12: una muestra por donante

m = (adata.obs["cell_type"] == "CD14_Mono") & (adata.obs["cytokine"] == "IL-4")
adata[m, ["CXCL9"]].to_df().sort_values("CXCL9", ascending=False)
```

</details>

In [ ]:
# 🔧 tu código aquí

## 2.3 · ¿Qué es un conteo? Tamaños de librería

Un **conteo** es el número de **UMI** (*identificadores moleculares únicos*: un UMI = una molécula de mRNA original, tras eliminar los duplicados de PCR)
asignados a un gen. En nuestro pseudobulk, ese número está **sumado sobre todas las células** de la muestra.
Así, un conteo es un **entero no negativo** que depende de dos cosas:

1. cuánto se **expresó** el gen (la biología que queremos), y
2. cuánto **material** tenemos: cuántas células y con qué profundidad se secuenciaron (técnico).

### Los datos crudos de un vistazo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))

# (izquierda) todos los conteos de 150 muestras al azar
vals = adata.X[rng.choice(adata.n_obs, 150, replace=False)].ravel()
axes[0].hist(np.log10(vals + 1), bins=80, color="#2a78d6")
axes[0].set_xlabel("log10(conteo + 1)"); axes[0].set_ylabel("número de entradas (muestra, gen)")
axes[0].set_title(f"Conteos: {np.mean(vals == 0):.0%} ceros y valores de hasta {vals.max():,.0f}")

# (derecha) unos pocos genes acaparan buena parte de los conteos
gene_total = adata.var["total_counts"].values
order = np.argsort(gene_total)[::-1]
share = np.cumsum(gene_total[order]) / gene_total.sum() * 100
axes[1].plot(np.arange(1, len(share) + 1), share, color="#2a78d6", lw=2)
axes[1].set_xscale("log")
for k in (10, 100, 1000):
    axes[1].plot(k, share[k - 1], "o", color=UP_COLOR)
    axes[1].annotate(f"top {k:,} genes: {share[k - 1]:.0f}%", (k, share[k - 1]), xytext=(-8, 8), textcoords="offset points", ha="right", fontsize=8)
axes[1].set_xlabel("genes, ordenados por conteos totales"); axes[1].set_ylabel("% acumulado de todos los conteos")
axes[1].set_title("1,000 genes (de 18,120) acumulan la mitad de los conteos")
plt.tight_layout(); plt.show()

### Tamaño de librería: ¿cuántos conteos tiene cada muestra?

El **tamaño de librería** de una muestra es la suma de todos sus conteos: $L_i=\sum_j \text{conteo}_{ij}$. Comparemos los cuatro tipos celulares.

In [ ]:
adata.obs["lib_size"]      = adata.X.sum(axis=1)                              # conteos totales de cada muestra
adata.obs["umis_per_cell"] = adata.obs["lib_size"] / adata.obs["psbulk_n_cells"]

obs     = adata.obs
is_ctrl = (obs["treatment"] == "control").values

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

# (izquierda) tamaño de librería de cada muestra, por tipo celular  (círculos = muestras tratadas; anillos = control/PBS)
ax = axes[0]
for k, ct in enumerate(CELL_TYPES):
    m = (obs["cell_type"] == ct).values
    v, vp = obs.loc[m & ~is_ctrl, "lib_size"].values, obs.loc[m & is_ctrl, "lib_size"].values
    ax.scatter(k - 0.1 + rng.uniform(-0.2, 0.2, len(v)), v, s=7, alpha=0.4, linewidths=0, color=CT_COLORS[ct], marker=CT_MARKERS[ct])
    ax.scatter(k + 0.33 + rng.uniform(-0.06, 0.06, len(vp)), vp, s=30, facecolors="none", edgecolors="k", linewidths=0.9)
    ax.hlines(np.median(v), k - 0.4, k + 0.2, color="k", lw=2)
    ax.text(k - 0.1, 1.5e8, f"mediana\n{np.median(v) / 1e6:.1f} M", ha="center", fontsize=8)
ax.set_yscale("log"); ax.set_ylim(5e4, 6e8)
ax.set_xticks(range(4)); ax.set_xticklabels([CT_LABEL[c].replace(" ", "\n", 1) for c in CELL_TYPES])
ax.set_ylabel("tamaño de librería (conteos totales)"); ax.set_title("Tamaño de librería por tipo celular  (○ = controles PBS)")

# (derecha) el tamaño de librería sigue al número de células sumadas
ax = axes[1]
for ct in CELL_TYPES:
    m = (obs["cell_type"] == ct).values & ~is_ctrl
    ax.scatter(obs.loc[m, "psbulk_n_cells"], obs.loc[m, "lib_size"], s=7, alpha=0.4, linewidths=0, color=CT_COLORS[ct], marker=CT_MARKERS[ct], label=CT_LABEL[ct])
ax.scatter(obs.loc[is_ctrl, "psbulk_n_cells"], obs.loc[is_ctrl, "lib_size"], s=30, facecolors="none", edgecolors="k", linewidths=0.9, label="controles PBS")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("número de células en el pseudobulk"); ax.set_ylabel("tamaño de librería (conteos totales)")
ax.set_title("Tamaño de librería ≈ nº de células × UMI por célula"); ax.legend(markerscale=1.8, loc="upper left")
plt.tight_layout(); plt.show()

summary = obs[~is_ctrl].groupby("cell_type", observed=True)[["psbulk_n_cells", "umis_per_cell", "lib_size"]].median().round(0).astype(int)
summary.columns = ["células por muestra", "UMI por célula", "tamaño de librería"]
print("Mediana de las muestras tratadas:"); display(summary)
print(f"Los controles PBS son {obs.loc[is_ctrl, 'lib_size'].median() / obs.loc[~is_ctrl, 'lib_size'].median():.1f}× más profundos que una muestra tratada típica.")

**❓ ¿Por qué los cuatro tipos celulares tienen tamaños de librería tan distintos? ¿Y por qué las muestras de PBS son mucho mayores?**

<details>
<summary>💡 Respuesta</summary>

**Tamaño de librería = células × UMI por célula.**
* **Número de células.** Los linfocitos T CD4 son, con diferencia, el tipo celular más abundante en estas muestras (≈ 3,000 células por muestra) y las células NK son escasas (≈ 400). Cuantas más células sumamos, más conteos obtenemos.
* **Contenido de RNA por célula.** Los monocitos son células más grandes con más mRNA: ≈ 6,700 UMI por célula frente a ≈ 3,000 en los linfocitos. Por eso los monocitos alcanzan un tamaño de librería similar al de los T CD4 con solo ~40 % de las células.
* Los **controles PBS** son la suma de **6 pocillos** (~5× más células), así que sus librerías son mucho mayores.

Nada de esto es la biología que buscamos, así que antes de comparar muestras tenemos que eliminarlo.

</details>

## 2.4 · Normalización

### ¿Por qué? Un ejemplo

Imagina que quieres saber si la palabra *"citoquina"* se usa más en un artículo de 10 páginas o en una revisión de 100. Contar palabras no basta: la revisión tiene más palabras de *todo*.
Compararías **frecuencias** (apariciones por cada 10,000 palabras). Eso es exactamente lo que necesitamos para los genes.

Aquí tienes una versión de juguete con **el mismo donante**: un control PBS (suma de 6 pocillos, así que *profundo*: 5,000 conteos en total) y una muestra de IFN-γ (*superficial*: 1,000 conteos en total).
El gen **B** es un gen de respuesta a interferón que IFN-γ induce de verdad. Los demás genes no cambian.

In [ ]:
toy = pd.DataFrame({"PBS (profundo)":    [1000, 50, 2500, 1450],
                    "IFN-γ (superficial)": [ 200, 40,  500,  260]},
                   index=["gen A", "gen B (ISG)", "gen C", "gen D"])
toy["cociente crudo IFN-γ / PBS"] = (toy["IFN-γ (superficial)"] / toy["PBS (profundo)"]).round(2)
print("CONTEOS CRUDOS   (tamaños de librería: 5,000 vs 1,000)")
display(toy)

cpm_toy = toy.iloc[:, :2] / toy.iloc[:, :2].sum() * 1e6              # conteos por millón = conteo / tamaño de librería × 1,000,000
cpm_toy["cociente CPM IFN-γ / PBS"] = (cpm_toy["IFN-γ (superficial)"] / cpm_toy["PBS (profundo)"]).round(2)
print("CONTEOS POR MILLÓN (CPM)")
display(cpm_toy.round(0))

En conteos crudos **todos los genes parecen menores con IFN-γ, incluido el gen B de respuesta a interferón**, simplemente porque esa muestra es 5× más superficial.
Tras dividir por el tamaño de librería, los genes A, C y D no cambian y el gen **B sube 4×**: la respuesta correcta.

Los **Conteos Por Millón (CPM)** expresan cada gen como *partes por millón de las moléculas de la muestra*:

$$\text{CPM}_{ij} \;=\; \frac{\text{conteo}_{ij}}{L_i}\times 10^{6},\qquad L_i=\sum_{j}\text{conteo}_{ij}$$

### ¿Funciona con datos reales?

Si los CPM funcionan, un gen **constitutivo** (`ACTB`) ya *no* debería depender del tamaño de librería. Monocitos, todas las condiciones (incluidas las muestras profundas de PBS):

In [ ]:
counts = adata.X.copy()                                  # conteos crudos, guardados aparte
lib    = counts.sum(axis=1, keepdims=True)               # tamaño de librería de cada muestra (una columna de 4,224 números)
cpm    = counts / lib * 1e6                              # conteos por millón
print("Ahora cada muestra suma un millón:", np.allclose(cpm.sum(axis=1), 1e6, rtol=1e-4))

m_mono = (adata.obs["cell_type"] == "CD14_Mono").values
j = adata.var_names.get_loc("ACTB")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
for ax, vals, label in zip(axes, [counts[m_mono, j], cpm[m_mono, j]], ["conteo crudo", "CPM"]):
    ax.scatter(lib[m_mono, 0], vals, s=7, alpha=0.5, color=CT_COLORS["CD14_Mono"], linewidths=0)
    ax.set_xscale("log"); ax.set_yscale("log")
    r = np.corrcoef(np.log10(lib[m_mono, 0]), np.log10(vals + 1))[0, 1]
    ax.set_title(f"ACTB — {label}\ncorrelación con el tamaño de librería: r = {r:+.2f}")
    ax.set_xlabel("tamaño de librería de la muestra"); ax.set_ylabel(label)
plt.tight_layout(); plt.show()

Antes: los conteos de `ACTB` crecen con el tamaño de librería. Tras los CPM: la nube es plana: el gen es "constante", como debe ser.

### El logaritmo: un problema más, entre *genes*

Los CPM arreglan las **muestras**. Pero mira los **genes**: unos pocos se expresan a decenas de miles de CPM y la mayoría a unos pocos.
Las estadísticas y el PCA dependen de la **varianza**, así que veamos cómo se relaciona la varianza de un gen con su nivel de expresión:

In [ ]:
log_cpm = np.log2(cpm + 1)                                                  # log2(CPM + 1)

mean_cpm, var_cpm = cpm.mean(axis=0, dtype=np.float64),     cpm.var(axis=0, dtype=np.float64)
mean_log, var_log = log_cpm.mean(axis=0, dtype=np.float64), log_cpm.var(axis=0, dtype=np.float64)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ok = mean_cpm > 0
axes[0].hist(np.log10(mean_cpm[ok]), bins=70, color="#2a78d6")
axes[0].set_xlabel("log10(CPM medio del gen)"); axes[0].set_ylabel("número de genes")
axes[0].set_title("① Los genes abarcan 6 órdenes de magnitud")

ok = (mean_cpm > 0.01) & (var_cpm > 0)
axes[1].scatter(mean_cpm[ok], var_cpm[ok], s=3, alpha=0.25, color="#2a78d6", linewidths=0, rasterized=True)
xx = np.array([0.01, 1e5]); axes[1].plot(xx, xx ** 2 * 0.5, color="k", ls="--", lw=1.2, label="referencia: pendiente 2 (varianza ∝ media²)")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].legend(loc="upper left")
axes[1].set_xlabel("CPM medio"); axes[1].set_ylabel("varianza del CPM"); axes[1].set_title("② CPM: la varianza crece mucho con la expresión")

ok = var_log > 0
axes[2].scatter(mean_log[ok], var_log[ok], s=3, alpha=0.25, color="#2a78d6", linewidths=0, rasterized=True)
axes[2].set_xlabel("log2(CPM + 1) medio"); axes[2].set_ylabel("varianza de log2(CPM + 1)")
axes[2].set_title("③ log2(CPM + 1): la varianza ya no depende de la expresión")
plt.tight_layout(); plt.show()

top20_cpm = np.sort(var_cpm)[-20:].sum() / var_cpm.sum()
top20_log = np.sort(var_log)[-20:].sum() / var_log.sum()
print(f"Parte de la varianza TOTAL que acumulan los 20 genes más variables:  CPM: {top20_cpm:.0%}   →   log2(CPM+1): {top20_log:.1%}")

En CPM, la varianza crece de forma pronunciada con la media, aproximadamente con su **cuadrado** en los genes abundantes (panel ②): unos pocos genes muy abundantes decidirían cualquier análisis basado en la varianza (PCA, clustering, pruebas t).
El **logaritmo** comprime los valores grandes, de modo que la varianza ya no depende de lo abundante que sea el gen (panel ③):

$$\text{log-CPM}_{ij}=\log_2\bigl(\text{CPM}_{ij}+1\bigr)$$

Extra: el log₂ hace que los **cambios (fold change) sean simétricos** (duplicar es +1, reducir a la mitad es −1) y la *diferencia* de dos valores logarítmicos **es** el log fold change:

| fold change | ×4 | ×2 | ×1.41 | ×1 | ×0.71 | ×0.5 | ×0.25 |
|---|---|---|---|---|---|---|---|
| **log₂ fold change** | +2 | +1 | +0.5 | 0 | −0.5 | −1 | −2 |

**¿Por qué "+ 1"?** Porque log(0) no existe, y porque los números diminutos dan cocientes disparatados:

In [ ]:
ex = pd.DataFrame({"gene": ["casi silencioso", "silencioso → detectado", "moderado", "alto"],
                   "CPM control": [0.1, 0.0, 20.0, 500.0], "CPM tratado": [0.4, 0.5, 40.0, 1000.0]})
with np.errstate(divide="ignore"):
    ex["log2FC sin +1"] = np.log2(ex["CPM tratado"] / ex["CPM control"]).round(2)
ex["log2FC con +1"] = np.log2((ex["CPM tratado"] + 1) / (ex["CPM control"] + 1)).round(2)
display(ex)

Sin el pseudoconteo, un gen que pasa de 0.1 a 0.4 CPM (una diferencia de *una o dos moléculas*) parece subir 4× y un gen que parte de cero da infinito.
Con `+1` esos fold changes se encogen hacia 0, mientras que los genes moderados y altos no se ven afectados: *"no me fío de los cocientes de números diminutos"*.

### Aplicarlo a todo el objeto

A partir de ahora `adata.X` contiene **log₂(CPM + 1)**. Los conteos crudos no se pierden: viven en las **capas** (*layers*: más matrices con la misma forma que `X`).

In [ ]:
adata.layers["counts"] = counts            # conteos crudos
adata.layers["cpm"]    = cpm               # conteos por millón
adata.X                = log_cpm           # log2(CPM + 1): la matriz por defecto a partir de ahora
print(adata)
print(f"\nAhora X va de {adata.X.min():.1f} a {adata.X.max():.1f}  (los conteos crudos llegaban a {adata.layers['counts'].max():,.0f})")

## 2.5 · Filtrar genes poco expresados

Muchos de nuestros 18,120 genes apenas se detectan en células sanguíneas. Conservarlos es un problema porque:

1. sus conteos son casi **puro ruido de muestreo**: no contienen información;
2. sus fold changes son **poco fiables**: con 2 moléculas frente a 4, un "doblamiento" es solo mala suerte;
3. cada gen extra es **una prueba estadística más** que corregir en la Parte 4 (más pruebas → menos potencia).

### Un ejemplo claro: el mismo gen, medido dos veces

Simulamos dos **réplicas del mismo gen** (¡sin cambio real!), a cuatro niveles de expresión. Los conteos siguen una distribución de Poisson, el ruido mínimo de cualquier proceso de conteo.

In [ ]:
# 🔧 CAMBIA ESTO: otros niveles de expresión (conteos esperados), p. ej. [1, 5, 50, 500]
true_means = [2, 20, 200, 2000]                                    # número esperado de conteos del gen
fig, axes = plt.subplots(1, 4, figsize=(14, 2.9), sharex=True, sharey=True)
for ax, lam in zip(axes, true_means):
    rep1, rep2 = rng.poisson(lam, 5000), rng.poisson(lam, 5000)    # dos réplicas, 5,000 "genes" con la misma expresión
    lfc = np.log2((rep1 + 1) / (rep2 + 1))                         # fold change entre réplicas: la verdad es 0
    ax.hist(lfc, bins=np.linspace(-4, 4, 41), color="#2a78d6" if lam >= 200 else PBS_COLOR)
    ax.set_title(f"conteo esperado = {lam}\n{np.mean(np.abs(lfc) > 1):.0%} de los genes con |log2FC| > 1", fontsize=9.5)
    ax.set_xlabel("log2 fold change entre réplicas")
axes[0].set_ylabel("número de genes")
plt.tight_layout(); plt.show()

Con 2 conteos esperados, **una gran fracción de genes muestra un "cambio" >2× por puro azar**; con 2,000 conteos, ninguno. Los genes con pocos conteos producen **fold changes falsos**.

### La regla

Conservamos un gen si tiene **al menos 10 conteos en al menos 12 muestras**: 12 es el número de donantes, es decir, el tamaño de un grupo experimental.
Un gen que nunca alcanza eso en un grupo entero no se puede analizar en ningún sitio. (Es la idea de `filterByExpr` del paquete *edgeR*.)

In [ ]:
# 🔧 CAMBIA ESTO: un filtro más estricto o más laxo, y mira cuántos genes sobreviven
MIN_COUNT, MIN_SAMPLES = 10, 12          # ≥ 10 conteos en ≥ 12 muestras

def is_expressed(counts, min_count=MIN_COUNT, min_samples=MIN_SAMPLES):
    '''Para una matriz muestras × genes de conteos crudos: True para los genes con al menos `min_count` conteos en al menos `min_samples` muestras.'''
    return (counts >= min_count).sum(axis=0) >= min_samples

keep = is_expressed(adata.layers["counts"])

n_ok = (adata.layers["counts"] >= MIN_COUNT).sum(axis=0)           # para cada gen: ¿en cuántas muestras tiene ≥ 10 conteos?
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(np.log10(n_ok + 1), bins=60, color="#2a78d6")
ax.axvline(np.log10(MIN_SAMPLES + 1), color=UP_COLOR, ls="--"); ax.text(np.log10(MIN_SAMPLES + 1) - 0.05, ax.get_ylim()[1] * 0.9, "umbral: 12 muestras", color=UP_COLOR, ha="right")
ax.set_xlabel("log10(1 + nº de muestras con ≥ 10 conteos)"); ax.set_ylabel("número de genes")
ax.set_title(f"{keep.sum():,} de {len(keep):,} genes superan el filtro")
plt.tight_layout(); plt.show()

removed = adata.var_names[~keep]
print(f"Genes eliminados: {len(removed):,}   (solo aportan {adata.layers['counts'][:, ~keep].sum() / adata.layers['counts'].sum():.3%} de los conteos)")
print("Algunos de ellos:", list(rng.choice(removed, 10, replace=False)))

Los genes eliminados están prácticamente **ausentes** en las células sanguíneas (receptores olfativos, genes de otros tejidos…) y aportan casi nada de los conteos: quitarlos no pierde nada. Ahora aplicamos el filtro:

In [ ]:
adata = adata[:, keep].copy()            # seleccionar los genes; X, las capas y var se cortan a la vez
print(adata)

## 2.6 · Mapas de calor: antes y después de normalizar

Un mapa de calor muestra muchos genes (filas) en muchas muestras (columnas). Dibujaremos los **mismos genes** en las **mismas muestras** dos veces:
con los **conteos crudos** y con los **valores normalizados**, y veremos qué cambia la normalización.

**Muestras (columnas):** los 4 tipos celulares × 5 condiciones (el control PBS + 4 citoquinas que desencadenan programas muy distintos) × 12 donantes = 240 muestras.
La barra gris superior muestra el **tamaño de librería** de cada columna.

**Un truco más, solo para mapas de calor: el z-score.** Cada gen tiene su propio nivel basal, así que colorear los valores crudos solo mostraría qué genes son abundantes.
Por eso **escalamos cada gen (fila) a lo largo de las muestras**: restamos su media y dividimos por su desviación estándar, de modo que

$$z_{gi}=\frac{x_{gi}-\bar{x}_{g}}{s_{g}}\qquad\Rightarrow\qquad \text{rojo = por encima de la media del gen, azul = por debajo}$$

In [ ]:
#@title Ayuda para mapas de calor (ejecutar una vez, no hace falta leer) { display-mode: "form" }
CONDITIONS = ["PBS", "IFN-gamma", "IL-4", "IL-1-beta", "IL-2"]

def zscore_rows(M):
    '''z-score de cada FILA (gen) a lo largo de sus columnas (muestras).'''
    return (M - M.mean(axis=1, keepdims=True)) / M.std(axis=1, ddof=1, keepdims=True)

def before_after_heatmap(adata, gene_sets, conditions=CONDITIONS, title=""):
    cols   = [f"{d}|{ct}|{cy}" for ct in CELL_TYPES for cy in conditions for d in DONORS]      # ordenadas: tipo celular > condición > donante
    genes  = [g for gs in gene_sets.values() for g in gs if g in adata.var_names]
    sub    = adata[cols, genes]
    panels = [("ANTES: conteos crudos", sub.layers["counts"].T), ("DESPUÉS: log2(CPM + 1)", sub.X.T)]
    lib    = np.log10(adata.obs.loc[cols, "lib_size"].values)[None, :]                         # tamaño de librería de cada columna
    ncol   = len(conditions) * len(DONORS)                                                      # columnas por tipo celular
    sizes  = [len([g for g in gs if g in adata.var_names]) for gs in gene_sets.values()]

    fig = plt.figure(figsize=(14, 0.26 * len(genes) + 2.6))
    gs  = fig.add_gridspec(3, 3, height_ratios=[0.9, 0.9, len(genes)], width_ratios=[1, 1, 0.025],
                           hspace=0.05, wspace=0.07, top=0.88, bottom=0.14, left=0.06, right=0.95)
    for c, (name, M) in enumerate(panels):
        ax_ct, ax_lib, ax = (fig.add_subplot(gs[r, c]) for r in range(3))
        # barra superior: tipos celulares
        for k, ct in enumerate(CELL_TYPES):
            ax_ct.add_patch(plt.Rectangle((k * ncol - 0.5, 0), ncol, 1, color=CT_COLORS[ct], lw=0))
            ax_ct.text(k * ncol + ncol / 2 - 0.5, 0.5, CT_SHORT[ct], ha="center", va="center", fontsize=9, fontweight="bold",
                       color="black" if ct == "NK_CD56low" else "white")
        ax_ct.set_xlim(-0.5, len(cols) - 0.5); ax_ct.set_ylim(0, 1); ax_ct.axis("off")
        ax_ct.set_title(name, fontsize=11.5, pad=8)
        # segunda barra: tamaño de librería de cada muestra (oscuro = profundo)
        ax_lib.imshow(lib, aspect="auto", cmap="Greys", interpolation="nearest")
        ax_lib.set_xticks([]); ax_lib.set_yticks([0] if c == 0 else [])
        if c == 0: ax_lib.set_yticklabels(["tamaño de\nlibrería"], fontsize=7)
        for sp in ax_lib.spines.values(): sp.set_visible(False)
        # el mapa de calor: z-score de cada gen a lo largo de las muestras
        im = ax.imshow(zscore_rows(M), aspect="auto", cmap=DIVERGING, vmin=-2.5, vmax=2.5, interpolation="nearest")
        ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes if c == 0 else [], fontsize=8)
        ax.set_xticks([(k + 0.5) * len(DONORS) - 0.5 for k in range(len(cols) // len(DONORS))])
        ax.set_xticklabels(list(conditions) * len(CELL_TYPES), rotation=90, fontsize=7)
        for b in np.cumsum(sizes)[:-1] - 0.5: ax.axhline(b, color="white", lw=2)                # grupos de genes
        for k in range(1, len(CELL_TYPES)): ax.axvline(k * ncol - 0.5, color="white", lw=2)      # tipos celulares
        for sp in ax.spines.values(): sp.set_visible(False)
    fig.colorbar(im, cax=fig.add_subplot(gs[2, 2]), label="z-score del gen")
    fig.suptitle(title, y=0.985, fontweight="bold", fontsize=11)
    plt.show()

### Marcadores de tipo celular

In [ ]:
# 🔧 CAMBIA ESTO: añade o sustituye genes (cualquier símbolo de gen del conjunto de datos)
CELL_TYPE_MARKERS = {
    "monocitos":   ["LYZ", "CD14", "S100A9", "FCN1"],
    "linfocitos T": ["CD3E", "CD3D"],
    "T CD8":        ["CD8A", "CD8B"],
    "células NK":   ["NKG7", "GNLY", "KLRF1"],
}
before_after_heatmap(adata, CELL_TYPE_MARKERS, title="Marcadores de tipo celular  ·  filas: genes, columnas: 240 muestras (12 donantes × 5 condiciones × 4 tipos celulares)")

**❓ Mira la barra gris (tamaño de librería: oscuro = profundo) y compara los dos paneles.**
**ANTES:** ¿qué columnas son las más brillantes, sea cual sea el gen? **DESPUÉS:** ¿sigue siendo así? ¿Puedes leer ya los cuatro tipos celulares?

<details>
<summary>💡 Respuesta</summary>

* **Antes**, las columnas más brillantes son siempre las de **PBS** (las oscuras en la barra de tamaño de librería), en todos los tipos celulares: la imagen muestra *cuántos conteos tiene cada muestra*, no qué son las células, y las demás condiciones del mismo tipo celular se ven pálidas.
* **Después**, el brillo del PBS desaparece. Cada tipo celular muestra un patrón uniforme en sus cinco condiciones y sus propios marcadores: `LYZ`, `CD14`, `S100A9`, `FCN1` en monocitos, `CD3E`/`CD3D` en linfocitos T, `CD8A`/`CD8B` en T CD8, `NKG7`/`GNLY`/`KLRF1` en células NK.

</details>

### Genes de la respuesta inmune

Cuatro citoquinas, cuatro vías de señalización, cuatro conjuntos de genes diana. ¿Qué muestras encienden qué genes?

In [ ]:
# 🔧 CAMBIA ESTO: añade tus genes inmunes favoritos; las 5 condiciones mostradas son CONDITIONS (ayuda oculta arriba)
IMMUNE_GENES = {
    "IFN-γ / STAT1":     ["STAT1", "IRF1", "GBP1", "IDO1", "CXCL9", "CXCL10"],
    "IL-4 / STAT6":      ["ALOX15", "CCL17", "FCER2"],
    "IL-1β / NF-κB":     ["IL6", "CXCL8", "CCL2"],
    "IL-2 / STAT5":      ["IL2RA", "GZMB", "CISH"],
}
before_after_heatmap(adata, IMMUNE_GENES, title="Genes de la respuesta inmune  ·  PBS y cuatro citoquinas, en los cuatro tipos celulares")

**❓ ¿Qué citoquina activa qué bloque de genes? ¿En qué tipo celular? ¿Hay alguna respuesta compartida por varios tipos celulares?**

<details>
<summary>💡 Respuesta</summary>

* **IL-4** → `ALOX15`, `CCL17`, `FCER2`: casi solo en **monocitos**. (IL-4 también *reduce* los genes de interferón: el bloque azul de IL-4 en las filas superiores.)
* **IFN-γ** → los genes estimulados por interferón (`GBP1`, `IDO1`, `CXCL9`, `CXCL10`…), más fuertes en **monocitos**. `STAT1`, `IRF1` y `GBP1` también suben en linfocitos T y NK: un núcleo *compartido*.
* **IL-1β** → genes inflamatorios (`IL6`, `CXCL8`, `CCL2`) en **monocitos**. IL-1β e IL-2 también aumentan genes de interferón en monocitos, probablemente por liberación secundaria de citoquinas (el artículo lo señala para IL-2).
* **IL-2** → `IL2RA`, `GZMB`, `CISH` en **NK y linfocitos T**, los linfocitos que llevan el receptor de IL-2.

El mapa de calor normalizado muestra **biología**; el crudo muestra **profundidad de secuenciación**.

</details>

> ⚠️ **Los z-scores son para visualización** (y clustering). Eliminan los niveles absolutos y exageran genes que son planos pero ruidosos.
> No calcules nunca fold changes a partir de z-scores: en la Parte 4 usamos diferencias de log₂(CPM + 1).

## 2.7 · Resumen

| Paso | Qué | Soluciona | Dónde se guarda |
|:--:|---|---|---|
| 1 | **CPM** = conteo / tamaño de librería × 10⁶ | tamaños de librería distintos (**entre muestras**) | `adata.layers["cpm"]` |
| 2 | **log₂(CPM + 1)** | enorme rango dinámico y relación media–varianza (**entre genes**) | `adata.X` |
| 3 | **filtrar** genes (≥ 10 conteos en ≥ 12 muestras) | ruido y demasiadas pruebas | filas de `adata` |
| 4 | **z-score por gen** | niveles basales que ocultan el patrón (**solo en mapas de calor**) | no se guarda |

`adata.layers["counts"]` = conteos crudos → `adata.layers["cpm"]` = CPM → `adata.X` = log₂(CPM + 1).

☕ **Descanso: 10 minutos.**

---
# Parte 3 · Análisis de Componentes Principales (PCA): un análisis para dominarlos a todos
⏱ *35 min* (≈ 10 min de diapositivas + 25 min aquí) — **Objetivo:** mirar miles de genes a la vez y descubrir **qué provoca la variación** entre muestras.

Cada una de nuestras 4,224 muestras se describe con ~15,500 números (uno por gen): imposible de dibujar. El **PCA** encuentra *unos pocos ejes nuevos* que conservan la mayor parte posible de la variación entre muestras.
Es **no supervisado** (no sabe nada de citoquinas, donantes ni tipos celulares) así que, para descubrir qué *significa* una PC, **coloreamos las muestras según los metadatos** y vemos qué coloración se alinea con los ejes.

| Paso | Qué | Tiempo |
|:--:|---|:--:|
| **3.1** | El principio con 2 genes: **rotación, SVD, proyección** | 6 min |
| **3.2** | PCA de **todas las muestras** | 4 min |
| **3.3** | Un gráfico, **cuatro coloraciones**: tipo celular, donante, citoquina, control vs tratado | 5 min |
| **3.4** | ¿Qué significa cada PC? **R²** y **cargas (loadings)** | 4 min |
| **3.5** | **Acercarse** a un tipo celular | 4 min |
| **3.6** | 🔧 Te toca | 2 min |

In [ ]:
#@title 🏃 Celda de rescate: marca la casilla SOLO si la Parte 2 no se ejecutó hasta el final (reconstruye `adata` de una vez) { display-mode: "form" }
REBUILD_ADATA = False  #@param {type:"boolean"}

if REBUILD_ADATA:
    MIN_COUNT, MIN_SAMPLES = 10, 12
    def is_expressed(counts, min_count=MIN_COUNT, min_samples=MIN_SAMPLES):
        return (counts >= min_count).sum(axis=0) >= min_samples

    # preparación (2.1)
    adata = adata_raw[~((adata_raw.obs["cytokine"] == "PBS") & adata_raw.obs["bc1_well"].notna()).values].copy()
    if "psbulk_props" in adata.layers:
        del adata.layers["psbulk_props"]
    obs = adata.obs[["donor", "cytokine", "treatment", "cell_type", "psbulk_n_cells"]].copy()
    cytokines = ["PBS"] + sorted(c for c in obs["cytokine"].astype(str).unique() if c != "PBS")
    obs["donor"]     = pd.Categorical(obs["donor"].astype(str), categories=DONORS)
    obs["cytokine"]  = pd.Categorical(obs["cytokine"].astype(str), categories=cytokines)
    obs["cell_type"] = pd.Categorical(obs["cell_type"].astype(str), categories=CELL_TYPES)
    obs["treatment"] = pd.Categorical(np.where(obs["cytokine"] == "PBS", "control", "treated"), categories=["control", "treated"])
    obs.index = (obs["donor"].astype(str) + "|" + obs["cell_type"].astype(str) + "|" + obs["cytokine"].astype(str)).values
    adata.obs = obs
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    adata.X = X.astype(np.float32)
    adata.var = pd.DataFrame(index=adata.var_names)
    adata.var["total_counts"]       = adata.X.sum(axis=0)
    adata.var["n_samples_detected"] = (adata.X > 0).sum(axis=0)
    # tamaños de librería, CPM, log2(CPM + 1) (2.3-2.4)
    counts = adata.X.copy()
    lib = counts.sum(axis=1, keepdims=True)
    cpm = counts / lib * 1e6
    adata.obs["lib_size"]      = lib[:, 0]
    adata.obs["umis_per_cell"] = adata.obs["lib_size"] / adata.obs["psbulk_n_cells"]
    adata.layers["counts"], adata.layers["cpm"] = counts, cpm
    adata.X = np.log2(cpm + 1)
    # filtro (2.5)
    adata = adata[:, is_expressed(adata.layers["counts"])].copy()
    print("✅ adata reconstruido:", adata.shape)
else:
    print("Nada que hacer: usamos el `adata` de la Parte 2.")

### Parámetros de esta parte

In [ ]:
# 🔧 CAMBIA ESTO: todos los parámetros de esta parte
N_HVG = 2000                                                                       # número de genes altamente variables usados para el PCA
PCA_CELL_TYPE = "CD14_Mono"                                                        # tipo celular del acercamiento de 3.5: "CD4_T_cell", "CD8_T_cell", "NK_CD56low" o "CD14_Mono"
CYTOKINES_PANEL = ["IFN-gamma", "IL-4", "TNF-alpha", "IL-10", "IL-6", "GM-CSF"]   # citoquinas destacadas en los gráficos (cualquiera de las 87; hasta 7)
assert len(CYTOKINES_PANEL) <= 7, "Como máximo 7 citoquinas (los colores se vuelven difíciles de distinguir)"
print(adata)

In [ ]:
#@title Ayudas para los gráficos de PCA (ejecutar una vez, no hace falta leer) { display-mode: "form" }
ES_COLOR_BY = {"cell_type": "tipo celular", "donor": "donante", "cytokine": "citoquina", "treatment": "control vs tratado", "log10_lib_size": "log10 del tamaño de librería"}
DONOR_HUES  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]      # 12 donantes = 6 colores × 2 formas de marcador
CYT_HUES    = DONOR_HUES + ["#008300"]
CYT_MARKERS = ["o", "s", "^", "D", "v", "P", "X"]
SEQ_POINTS  = LinearSegmentedColormap.from_list("seqp", ["#cde2fb", "#2a78d6", "#0d366b"])

def pca_scatter(ax, data, color_by, pcs=(0, 1), s=14, title=None):
    '''Diagrama de dispersión de data.obsm["X_pca"], coloreado por "cell_type", "donor", "cytokine", "treatment", una columna numérica de data.obs o un nombre de gen.'''
    sc, (a, b), var, obs = data.obsm["X_pca"], pcs, data.uns["pca_var"], data.obs
    kw = dict(linewidths=0.3, edgecolors="white")
    if color_by == "cell_type":
        for ct in CELL_TYPES:
            m = (obs["cell_type"] == ct).values
            if m.any(): ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.8, color=CT_COLORS[ct], marker=CT_MARKERS[ct], label=CT_LABEL[ct], **kw)
    elif color_by == "donor":
        for i, d in enumerate(DONORS):
            m = (obs["donor"] == d).values
            ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.85, color=DONOR_HUES[i % 6], marker="o" if i < 6 else "^", label=d, **kw)
    elif color_by == "cytokine":
        m = ~obs["cytokine"].isin(["PBS"] + CYTOKINES_PANEL).values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.6, color=NS_COLOR, alpha=0.5, linewidths=0, label="todas las demás citoquinas", zorder=1)
        for i, cy in enumerate(CYTOKINES_PANEL):
            m = (obs["cytokine"] == cy).values
            ax.scatter(sc[m, a], sc[m, b], s=s * 1.4, color=CYT_HUES[i], marker=CYT_MARKERS[i], label=cy, zorder=2, **kw)
        m = (obs["cytokine"] == "PBS").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 2.4, color="black", label="PBS", zorder=3, **kw)
    elif color_by == "treatment":
        m = (obs["treatment"] == "treated").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.8, color=NS_COLOR, alpha=0.6, linewidths=0, label=f"tratadas (citoquina): {m.sum():,}", zorder=1)
        ax.scatter(sc[~m, a], sc[~m, b], s=s * 2.4, color="black", label=f"control (PBS): {(~m).sum():,}", zorder=2, **kw)
    else:                                                                               # un número: una columna de obs o la expresión de un gen
        if color_by in obs.columns: values, label = obs[color_by].values.astype(float), color_by
        else:                       values, label = np.asarray(data[:, color_by].X).ravel(), f"{color_by}\nlog2(CPM+1)"
        order = np.argsort(values)                                                      # los valores más altos se dibujan encima
        pts = ax.scatter(sc[order, a], sc[order, b], c=values[order], cmap=SEQ_POINTS, s=s * 1.6, **kw)
        plt.colorbar(pts, ax=ax, shrink=0.8, pad=0.01, label=label)
    if color_by in ("cell_type", "donor", "cytokine", "treatment"):
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=7.5, markerscale=1.5, handletextpad=0.3, labelspacing=0.3,
                  ncol=2 if color_by == "donor" else 1)
    ax.set_xlabel(f"PC{a + 1} ({var[a]:.1f} %)"); ax.set_ylabel(f"PC{b + 1} ({var[b]:.1f} %)")
    ax.set_title(title or f"coloreado por {ES_COLOR_BY.get(color_by, color_by)}")

def pca_four_views(data, pcs=(0, 1), title="", views=("cell_type", "donor", "cytokine", "treatment")):
    '''Las mismas coordenadas del PCA, coloreadas de cuatro formas.'''
    fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
    for ax, cb in zip(axes.ravel(), views):
        pca_scatter(ax, data, cb, pcs=pcs)
    fig.suptitle(title, fontweight="bold", y=0.995)
    plt.tight_layout(); plt.show()

def r2_categorical(y, groups):
    '''Fracción de la varianza de y explicada por la pertenencia a un grupo (R² de un ANOVA de una vía).'''
    y, g = np.asarray(y, float), np.asarray(groups).astype(str)
    ss_tot = ((y - y.mean()) ** 2).sum()
    ss_between = sum(((y[g == k].mean() - y.mean()) ** 2) * (g == k).sum() for k in np.unique(g))
    return ss_between / ss_tot

def pc_r2_table(data, n_pc=8):
    '''R² de cada PC con cada variable de las muestras.'''
    sc, obs = data.obsm["X_pca"], data.obs
    loglib = np.log10(obs["lib_size"].astype(float).values)
    return pd.DataFrame({
        "tipo celular": [r2_categorical(sc[:, k], obs["cell_type"]) for k in range(n_pc)],
        "donante":    [r2_categorical(sc[:, k], obs["donor"]) for k in range(n_pc)],
        "citoquina": [r2_categorical(sc[:, k], obs["cytokine"]) for k in range(n_pc)],
        "control vs tratado": [r2_categorical(sc[:, k], obs["treatment"]) for k in range(n_pc)],
        "tamaño de librería": [np.corrcoef(sc[:, k], loglib)[0, 1] ** 2 for k in range(n_pc)],
    }, index=[f"PC{k + 1}" for k in range(n_pc)])

def plot_pc_summary(data, n_pc=8):
    '''Varianza explicada por cada PC (izquierda) y R² de cada PC con cada variable (derecha).'''
    var, R2 = data.uns["pca_var"][:n_pc], pc_r2_table(data, n_pc)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), gridspec_kw={"width_ratios": [1, 1.6]})
    axes[0].barh(range(n_pc), var, color="#2a78d6"); axes[0].invert_yaxis()
    axes[0].set_yticks(range(n_pc)); axes[0].set_yticklabels(R2.index); axes[0].set_xlabel("varianza explicada (%)"); axes[0].set_title("¿Cuánta varianza?")
    for i, v in enumerate(var): axes[0].text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=8)
    im = axes[1].imshow(R2.values, cmap=SEQUENTIAL, vmin=0, vmax=1, aspect="auto")
    for i in range(R2.shape[0]):
        for j in range(R2.shape[1]):
            axes[1].text(j, i, f"{R2.values[i, j]:.2f}", ha="center", va="center", fontsize=8.5, color="w" if R2.values[i, j] > 0.55 else "k")
    axes[1].set_xticks(range(R2.shape[1])); axes[1].set_xticklabels(R2.columns, rotation=20, ha="right"); axes[1].set_yticks(range(n_pc)); axes[1].set_yticklabels(R2.index)
    axes[1].set_title("¿Qué captura cada PC?  (R² con cada variable)"); plt.colorbar(im, ax=axes[1], shrink=0.85)
    plt.tight_layout(); plt.show()

def plot_loadings(data, loadings, genes, pcs=(0, 1, 2), n=8):
    '''Los genes con los mayores pesos positivos y negativos de cada PC.'''
    fig, axes = plt.subplots(1, len(pcs), figsize=(4.6 * len(pcs), 4.3))
    for ax, k in zip(np.atleast_1d(axes), pcs):
        s = pd.Series(loadings[:, k], index=genes).sort_values(); top = pd.concat([s.head(n), s.tail(n)])
        ax.barh(top.index, top.values, color=[DOWN_COLOR if v < 0 else UP_COLOR for v in top.values])
        ax.axvline(0, color="k", lw=0.6); ax.tick_params(axis="y", labelsize=8)
        ax.xaxis.set_major_locator(plt.MaxNLocator(4)); ax.set_xlabel("carga (loading)"); ax.set_title(f"PC{k + 1}  ({data.uns['pca_var'][k]:.1f} % de la varianza)")
    plt.tight_layout(); plt.show()

## 3.1 · El principio con 2 genes: rotación, SVD y proyección

Toma los **monocitos** y solo **dos genes**, `ISG15` y `MX1` (ambos genes estimulados por interferón). Cada muestra es un punto en un plano: *x* = `ISG15`, *y* = `MX1`.
Los dos genes suben y bajan juntos, así que la nube es **alargada**. El PCA encuentra esa dirección en tres pasos (las diapositivas mostraron el dibujo):

1. **Centrar** cada gen (restar su media): matriz $X_c$.
2. **SVD** (descomposición en valores singulares): $X_c = U\,S\,V^\top$. $V$ es una **rotación**: sus columnas (las *cargas*, *loadings*) son los nuevos ejes PC1, PC2, de longitud unidad y perpendiculares. $S$ contiene cuánto se estiran los datos a lo largo de cada eje.
3. **Proyección**: las coordenadas de cada muestra en los nuevos ejes, las *puntuaciones* (*scores*), son $X_c V$ (que es igual a $U S$).

In [ ]:
is_mono = (adata.obs["cell_type"] == "CD14_Mono").values
P = adata[is_mono, ["ISG15", "MX1"]].X.astype(np.float64)       # 1,056 muestras de monocitos × 2 genes, en log2(CPM + 1)

Pc = P - P.mean(axis=0)                                         # 1) centrar cada gen
U, S, Vt = np.linalg.svd(Pc, full_matrices=False)               # 2) SVD:  Pc = U · diag(S) · Vt
if Vt[0].sum() < 0: Vt[0], U[:, 0] = -Vt[0], -U[:, 0]           #    el signo de una PC es arbitrario: hacemos que PC1 apunte a "más de ambos genes"
V = Vt.T                                                        #    columnas de V = PC1, PC2 (las cargas)
scores = Pc @ V                                                 # 3) proyección de cada muestra sobre los nuevos ejes

var_pct = S ** 2 / np.sum(S ** 2) * 100                         #    varianza que lleva cada eje
print("Cargas V  (columnas = PC1, PC2;  filas = ISG15, MX1):\n", np.round(V, 3))
print(f"\nPC1 es el eje x girado {np.degrees(np.arctan2(V[1, 0], V[0, 0])):.0f}°  |  V es una rotación pura (VᵀV = identidad): {np.allclose(V.T @ V, np.eye(2))}")
print(f"Scores = U · S: {np.allclose(scores, U * S)}")
print(f"Varianza explicada:  PC1 = {var_pct[0]:.1f} %    PC2 = {var_pct[1]:.1f} %")

In [ ]:
is_ifn = adata.obs.loc[is_mono, "cytokine"].astype(str).str.startswith("IFN-").values           # los 8 interferones
mu = P.mean(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# (izquierda) los ejes originales, con los dos nuevos ejes encima y la proyección de algunas muestras sobre PC1
ax = axes[0]
for flag, col, lab in [(False, NS_COLOR, "otras citoquinas y PBS"), (True, UP_COLOR, "interferones (α, β, γ, λ, ε, ω)")]:
    ax.scatter(P[is_ifn == flag, 0], P[is_ifn == flag, 1], s=9, alpha=0.65, color=col, linewidths=0, label=lab)
for i in rng.choice(len(P), 14, replace=False):                                                 # proyección = bajar una perpendicular sobre PC1
    proj = mu + (Pc[i] @ V[:, 0]) * V[:, 0]
    ax.plot([P[i, 0], proj[0]], [P[i, 1], proj[1]], color="k", lw=0.9, ls=":")
    ax.scatter(*proj, s=14, color="k", zorder=5)
for k, nm in enumerate(["PC1", "PC2"]):                                                          # los nuevos ejes: longitud = ±1.5 desviaciones estándar
    d = V[:, k] * 1.5 * S[k] / np.sqrt(len(P) - 1)
    ax.annotate("", xy=mu + d, xytext=mu - d, arrowprops=dict(arrowstyle="<|-|>", color="k", lw=1.8), annotation_clip=False)
    ax.annotate(nm, xy=mu + d, xytext=(6, 4), textcoords="offset points", fontsize=11, fontweight="bold", annotation_clip=False)
ax.set_aspect("equal"); ax.set_xlabel("ISG15  (log2 CPM + 1)"); ax.set_ylabel("MX1  (log2 CPM + 1)"); ax.legend(loc="lower right")
ax.set_title("Ejes originales (punteado: proyección sobre PC1)")

# (derecha) los mismos puntos en los ejes rotados
ax = axes[1]
for flag, col in [(False, NS_COLOR), (True, UP_COLOR)]:
    ax.scatter(scores[is_ifn == flag, 0], scores[is_ifn == flag, 1], s=9, alpha=0.65, color=col, linewidths=0)
ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6); ax.set_aspect("equal")
ax.set_xlabel(f"PC1 ({var_pct[0]:.0f} % de la varianza)"); ax.set_ylabel(f"PC2 ({var_pct[1]:.0f} %)")
ax.set_title("Nuevos ejes: los mismos puntos, rotados")
plt.tight_layout(); plt.show()

El PCA es **solo una rotación** (más un centrado): todavía no hemos perdido nada. La **reducción de dimensionalidad** ocurre cuando conservamos solo los primeros ejes:
aquí **PC1 sola** resume casi toda la variación, y ambos genes tienen carga positiva, así que PC1 = *"nivel global de genes de interferón"*: las muestras de interferón (rojas) están en su extremo alto.
Con miles de genes la receta es **exactamente la misma**.

## 3.2 · PCA de todas las muestras

Con ~15,500 genes primero reducimos el ruido: la mayoría de los genes apenas varían entre muestras. Conservamos los **genes altamente variables (HVG)**: los `N_HVG` genes con mayor varianza de log₂(CPM + 1) entre muestras.
Después ejecutamos los mismos tres pasos, envueltos en una función:

In [ ]:
def pca_svd(M, n_components=10):
    '''PCA de una matriz muestras × genes: centrar → SVD → proyectar. Devuelve puntuaciones, % de varianza explicada y cargas.'''
    Mc = M.astype(np.float64) - M.mean(axis=0)                  # 1) centrar los genes
    U, S, Vt = np.linalg.svd(Mc, full_matrices=False)           # 2) SVD
    scores   = (U * S)[:, :n_components]                        # 3) proyección: coordenadas de las muestras en las PCs
    var_pct  = (S ** 2 / np.sum(S ** 2) * 100)[:n_components]   #    % de la varianza total que lleva cada PC
    loadings = Vt[:n_components].T                              #    pesos de los genes en cada PC (genes × PCs)
    return scores, var_pct, loadings

gene_var = adata.X.var(axis=0)
hvg      = np.argsort(gene_var)[::-1][:N_HVG]                   # índices de los genes más variables
print("Los 10 genes más variables:", adata.var_names[hvg[:10]].tolist())

scores, var_pct, loadings = pca_svd(adata.X[:, hvg])
adata.obs["log10_lib_size"] = np.log10(adata.obs["lib_size"].astype(float))    # (se usa para colorear los gráficos)
adata.obsm["X_pca"]  = scores                                   # AnnData guarda en `obsm` las anotaciones multidimensionales de las muestras
adata.uns["pca_var"] = var_pct

sk = PCA(n_components=10).fit(adata.X[:, hvg])                  # comprobación: el PCA de scikit-learn da las mismas varianzas
print("Igual que scikit-learn:", np.allclose(var_pct, sk.explained_variance_ratio_ * 100, atol=1e-3))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(range(1, 11), var_pct, color="#2a78d6")
for i, v in enumerate(var_pct[:5]): ax.text(i + 1, v + 1, f"{v:.1f} %", ha="center", fontsize=8.5)
ax.set_xticks(range(1, 11)); ax.set_xlabel("componente principal"); ax.set_ylabel("varianza explicada (%)")
ax.set_title(f"Gráfico de sedimentación (scree): las 4 primeras PCs acumulan {var_pct[:4].sum():.0f} % de la varianza")
plt.tight_layout(); plt.show()

## 3.3 · Un gráfico, cuatro coloraciones

Las coordenadas no cambian nunca, solo el color. Las muestras son **tipos celulares**, **donantes**, **citoquinas** (el control PBS en negro, las citoquinas de `CYTOKINES_PANEL` en color, todas las demás en gris) y
**control vs tratado**.

In [ ]:
pca_four_views(adata, pcs=(0, 1), title=f"Las {adata.n_obs:,} muestras · PCA de los {N_HVG:,} genes más variables · PC1 vs PC2")

**❓ Preguntas**
1. ¿Qué PCs separan los **tipos celulares** y cuánta varianza acumulan? ¿Qué dos tipos celulares son los más parecidos?
2. ¿Separan los **donantes**, las **citoquinas** o **control vs tratado** en PC1–PC2?
3. ¿Significa eso que las citoquinas no tienen efecto sobre las células?

<details>
<summary>💡 Respuestas</summary>

1. **PC1 (66 %)** separa monocitos de linfocitos; **PC2 (17 %)** separa células NK de linfocitos T. Los T CD4 y CD8 son la pareja más cercana (se solapan en estas dos PCs).
2. **No.** Dentro de cada grupo de tipo celular, donantes, citoquinas y controles están mezclados; los puntos negros de PBS están *dentro* de la nube de muestras tratadas.
3. **¡No!** El PCA muestra primero las fuentes de variación **más grandes**, y la diferencia entre un monocito y un linfocito T es mucho mayor que el efecto de cualquier citoquina. Los efectos de citoquina y de donante viven en PCs *posteriores* o dentro de cada tipo celular (próximos pasos).

</details>

## 3.4 · ¿Qué significa cada PC?

Los colores dan una impresión; los números dan certeza. Para cada PC calculamos la **R²** con cada variable: la fracción de la varianza de la PC explicada por la variable
(para una categoría: varianza entre grupos / varianza total, como en un ANOVA; para el tamaño de librería: la correlación al cuadrado).

In [ ]:
plot_pc_summary(adata, n_pc=8)

A continuación, una PC es una suma ponderada de genes. Las **cargas (loadings)** indican qué genes tienen los mayores pesos, es decir, **qué genes definen el eje**
(el signo de una PC es arbitrario: cambiar todos los signos da un PCA equivalente).

In [ ]:
plot_loadings(adata, loadings, adata.var_names[hvg], pcs=(0, 1, 2))

**❓ Preguntas**
1. Usando la tabla de R² y las cargas: ¿qué es **PC1**? ¿**PC2**? ¿**PC3**? ¿Tienen sentido los genes?
2. ¿Cuál es la primera PC con alguna señal de **citoquina**? ¿Es una PC de varianza alta o baja? ¿Qué dice eso del tamaño de los efectos de las citoquinas?
3. PC2 tiene una R² de ~0.4 con el **tamaño de librería**. ¿Debemos preocuparnos?

<details>
<summary>💡 Respuestas</summary>

1. **PC1**: genes de monocitos (`FPR3`, `CYBB`, `CD36`…) frente a genes de linfocitos (`CD96`, `SKAP1`, `CD247`…). **PC2**: genes NK (`NCR1`, `KLRF1`, `GNLY`, `NCAM1`) frente a genes de T vírgenes (`CCR7`, `LEF1`, `TSHZ2`). **PC3** es una *mezcla*: genes de respuesta a interferón (`ISG15`, `IFIT1/2/3`, `RSAD2`) a un lado y **genes del cromosoma Y** (`UTY`, `USP9Y`, `KDM5D`) al otro. Un eje de tipo citoquina y un eje de donante/sexo en la misma PC.
2. **PC3** (R² citoquina ≈ 0.13, donante ≈ 0.72). Solo contiene ~3 % de la varianza: los efectos de las citoquinas son **pequeños comparados con las diferencias entre tipos celulares**.
3. No realmente: las muestras NK tienen simplemente librerías más pequeñas (Parte 2), así que PC2, que separa NK de T, se correlaciona con el tamaño de librería. Es el *tipo celular* disfrazado (un factor de confusión), no un artefacto de profundidad. La normalización de la Parte 2 hizo su trabajo.

</details>

## 3.5 · Acercarse: PCA dentro de un tipo celular

Las diferencias entre tipos celulares lo ocultan todo. Un remedio habitual es analizar **un tipo celular cada vez**: seleccionamos el objeto, volvemos a elegir los HVG y repetimos el PCA. Hagámoslo con el tipo celular de `PCA_CELL_TYPE` (por defecto los **monocitos CD14**, los más sensibles; los comentarios de abajo describen el valor por defecto).

In [ ]:
mono = adata[adata.obs["cell_type"] == PCA_CELL_TYPE].copy()                   # un nuevo AnnData con las 1,056 muestras de un tipo celular
hvg_m = np.argsort(mono.X.var(axis=0))[::-1][:N_HVG]                            # HVG del tipo celular elegido
mono.obsm["X_pca"], mono.uns["pca_var"], load_m = pca_svd(mono.X[:, hvg_m])

pca_four_views(mono, pcs=(0, 1), title=f"Solo {CT_LABEL[PCA_CELL_TYPE]} · PC1 vs PC2", views=("donor", "cytokine", "treatment", "log10_lib_size"))

**❓ ¿Qué ha cambiado respecto al gráfico de todos los tipos celulares?**

<details>
<summary>💡 Respuesta</summary>

* Los **donantes** ahora forman grupos claros: las *diferencias basales entre personas* son grandes dentro de un tipo celular. El Donor 2 queda apartado (el artículo observa lo mismo en sus monocitos, Fig. 2k).
* Las **citoquinas** siguen sin separarse en PC1–PC2: los efectos individuales son menores que las diferencias entre donantes. La nube gris está formada por 87 citoquinas × 12 donantes, así que solo se distinguen los colores de las destacadas. Por eso, en la Parte 4, cada citoquina se compara con el PBS del **mismo donante** (un diseño *pareado*).
* **Control vs tratado**: las muestras de PBS no se diferencian del resto en estas PCs.

</details>

Los efectos de las citoquinas aparecen en PCs *posteriores*. Veamos PC3 vs PC4 y de qué están hechas estas PCs:

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 5))
pca_scatter(ax, mono, "cytokine", pcs=(2, 3), title=f"{CT_LABEL[PCA_CELL_TYPE]} · PC3 vs PC4, coloreado por citoquina")
plt.tight_layout(); plt.show()

plot_loadings(mono, load_m, mono.var_names[hvg_m], pcs=(2, 3))        # los genes detrás de PC3 y PC4

Ahora las citoquinas sí se separan: **IL-4** (cuadrados naranjas) se aleja de la nube hacia PC4 baja, cuyo extremo negativo contiene genes diana de IL-4 (`ALOX15`, `WNT5B`, `FN1`); **GM-CSF** (cruces moradas) va hacia PC3 alta (`CD1B`, `CD1E`, `CCL24`, `CLEC5A`, `INHBA`); **IL-10** (rombos amarillos) hacia PC3 baja.
El extremo bajo de PC3 también contiene **genes del cromosoma Y** (`UTY`, `KDM5D`): un eje de citoquina y un eje de sexo en la misma PC, como antes.
*Cada citoquina mueve las muestras en su propia dirección*, mezclada con efectos de donante.

## 3.6 · 🔧 Te toca

Cambia los parámetros y vuelve a ejecutar la celda: `COLOR_BY` puede ser `"donor"`, `"cytokine"`, `"treatment"`, `"log10_lib_size"` o **cualquier símbolo de gen** (`"STAT1"`, `"IL1B"`, `"UTY"`…). El gráfico usa el PCA de `PCA_CELL_TYPE` (cámbialo en la celda de parámetros de esta parte y vuelve a ejecutar desde ahí).

In [ ]:
# 🔧 CAMBIA ESTO: por qué colorear y qué PCs dibujar
COLOR_BY = "UTY"        # "donor" | "cytokine" | "treatment" | "log10_lib_size" | cualquier símbolo de gen
PCS      = (2, 3)       # qué PCs en los ejes x e y, p. ej. (0, 1) para PC1 vs PC2

fig, ax = plt.subplots(figsize=(7.5, 5.3))
pca_scatter(ax, mono, COLOR_BY, pcs=PCS, s=34)
plt.tight_layout(); plt.show()

> 🔧 **Tareas**
> 1. Colorea por `UTY` (un gen del cromosoma Y). Con `PCS = (0, 1)` y con `(2, 3)`: ¿qué PC separa los donantes varones de las mujeres? *(el artículo indica 6 varones y 6 mujeres)*
> 2. Colorea por `log10_lib_size`. ¿Hay un gradiente suave a lo largo de un eje? ¿Sería una señal de alarma?
> 3. Cambia `CYTOKINES_PANEL` (en *Parámetros de esta parte*, y vuelve a ejecutar esa celda) por seis citoquinas de la familia de interferones, p. ej. `["IFN-alpha1", "IFN-beta", "IFN-gamma", "IFN-lambda1", "IFN-omega", "IL-27"]`, y colorea por `"cytokine"`. ¿Cuáles se agrupan?
> 4. Prueba `N_HVG = 200` y `N_HVG = 10000` y recalcula (3.5). ¿Qué tan robusta es la imagen?

## 3.7 · Resumen

* PCA = **rotación** de los datos hacia ejes de varianza decreciente (**SVD**); **puntuaciones (scores)** = proyección de las muestras, **cargas (loadings)** = pesos de los genes.
* Es **no supervisado**: colorear por metadatos te dice qué significa cada PC; la tabla de **R²** y las **cargas** lo hacen cuantitativo.
* La normalización decide lo que ve el PCA (Parte 2); sin ella, las primeras PCs reflejarían la profundidad de secuenciación.
* La variación más fuerte es el **tipo celular**; después los efectos de **donante**; los efectos de **citoquina** son reales pero menores, y aparecen dentro de un tipo celular o en PCs posteriores.
* 👉 Siguiente: hacer la pregunta gen a gen, **pareando por donante**: expresión diferencial.

---
# Parte 4 · Expresión diferencial: células tratadas vs no tratadas
⏱ *35 min* (≈ 10 min de diapositivas + 25 min aquí) — **Objetivo:** encontrar los genes que **cambian** al añadir una citoquina, decir **con qué seguridad** y comparar dos citoquinas.

El PCA mostró que las citoquinas mueven las muestras en direcciones particulares. La expresión diferencial plantea la pregunta **gen a gen**:

> *Para el gen g, ¿es su expresión con la citoquina distinta de su expresión con PBS, más de lo que esperaríamos por azar?*

| Paso | Pregunta | Herramienta | Tiempo |
|:--:|---|---|:--:|
| **4.1** | ¿Cómo de grande es el cambio y es consistente entre donantes? | diferencias pareadas, **log₂ fold change**, una **prueba t pareada** a mano | 6 min |
| **4.2** | …para todos los genes, como se debe | **DESeq2** (`pydeseq2`) sobre los conteos crudos | 10 min |
| **4.3** | Hemos hecho ~13,000 pruebas… | **corrección por pruebas múltiples** (Benjamini–Hochberg, **FDR**) | 5 min |
| **4.4** | ¿Qué genes llamamos *diferencialmente expresados*? | umbrales + **gráfico de volcán** | 4 min |
| **4.5** | ¿Cómo se diferencian dos citoquinas? | DESeq2 con **dos citoquinas** en uno o dos tipos celulares | 8 min |
| **4.6** | 🔧 Te toca | | 2 min |

> Si `adata` no existe (reiniciaste la sesión), ejecuta la **celda de rescate de la Parte 3**.

In [ ]:
# 🔧 CAMBIA ESTO: la comparación de las secciones 4.1-4.4 (cualquiera de las 87 citoquinas, cualquiera de los 4 tipos celulares)
CYTOKINE        = "IFN-gamma"      # la citoquina que comparamos con el control PBS
CELL_TYPE       = "CD14_Mono"      # "CD4_T_cell", "CD8_T_cell", "NK_CD56low" o "CD14_Mono"
EXAMPLE_GENE    = "CXCL9"          # un gen que esperamos que responda a IFN-γ (usa "ALOX15" para IL-4)
CONTROL_GENE    = "ACTB"           # un gen constitutivo: NO esperamos respuesta
PADJ_THRESH     = 0.05             # significación: valor p ajustado (FDR) por debajo de esto…
LFC_THRESH      = 0.5              # …y |log2 fold change| por encima de esto (0.5 ≈ un cambio de 1.4×)

# 🔧 CAMBIA ESTO: la comparación de la sección 4.5 (dos citoquinas, en uno o dos tipos celulares)
CYTOKINE_A         = "IFN-gamma"
CYTOKINE_B         = "IL-4"
CELL_TYPES_COMPARE = ["CD14_Mono", "CD4_T_cell"]     # uno o dos de los cuatro tipos celulares

print(f"4.1-4.4: {CYTOKINE} vs PBS en {CT_LABEL[CELL_TYPE]}  ·  gen de ejemplo {EXAMPLE_GENE}")
print(f"4.5    : {CYTOKINE_A} vs {CYTOKINE_B} en {[CT_LABEL[c] for c in CELL_TYPES_COMPARE]}")

In [ ]:
#@title Ayudas para los gráficos de DE (ejecutar una vez, no hace falta leer) { display-mode: "form" }
try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except ImportError:
    HAS_ADJUSTTEXT = False
CYTOKINE_COLOR = "#eb6834"

def paired_plot(ax, x_pbs, x_cyt, gene, cytokine):
    '''Una línea por donante: PBS → citoquina. Rojo = mayor con la citoquina, azul = menor.'''
    for a, b in zip(x_pbs, x_cyt):
        ax.plot([0, 1], [a, b], color=UP_COLOR if b > a else DOWN_COLOR, lw=1.2, alpha=0.7, zorder=1)
    ax.scatter(np.zeros(len(x_pbs)), x_pbs, s=30, color=PBS_COLOR, zorder=2)
    ax.scatter(np.ones(len(x_cyt)), x_cyt, s=30, color=CYTOKINE_COLOR, zorder=2)
    ax.plot([0, 1], [x_pbs.mean(), x_cyt.mean()], color="k", lw=3, marker="D", ms=7, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["PBS", cytokine]); ax.set_xlim(-0.35, 1.35)
    ax.set_ylabel("log2(CPM + 1)"); ax.set_title(f"{gene}\ndiferencia media = {np.mean(x_cyt - x_pbs):+.2f}")

def volcano(ax, de, title, n_label=16, xlabel="log2 fold change"):
    '''Gráfico de volcán: tamaño del efecto (x) frente a evidencia (y); se etiquetan los DEG más fuertes.'''
    d = de[de["tested"]].copy(); d["nlp"] = -np.log10(d["padj"].clip(lower=1e-300))
    for k, col, size, alpha in [("ns", NS_COLOR, 6, 0.5), ("down", DOWN_COLOR, 14, 0.8), ("up", UP_COLOR, 14, 0.8)]:
        sub = d[d["direction"] == k]
        ax.scatter(sub["log2fc"], sub["nlp"], s=size, color=col, alpha=alpha, linewidths=0, rasterized=True)
    for x in (-LFC_THRESH, LFC_THRESH): ax.axvline(x, color="#999", lw=0.8, ls="--")
    ax.axhline(-np.log10(PADJ_THRESH), color="#999", lw=0.8, ls="--")
    sig = d[d["direction"] != "ns"].copy()                             # etiquetar los DEG con mejor combinación de tamaño de efecto y evidencia
    sig["score"] = sig["nlp"] * sig["log2fc"].abs()
    ax.margins(y=0.12); x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()                   # mantener las etiquetas fuera de los dos contadores (esquinas superiores)
    corner = (sig["nlp"] > y0 + 0.85 * (y1 - y0)) & ((sig["log2fc"] < x0 + 0.3 * (x1 - x0)) | (sig["log2fc"] > x1 - 0.3 * (x1 - x0)))
    sig = sig[~corner]
    top = pd.concat([sig[sig["direction"] == "up"].nlargest(n_label // 2, "score"), sig[sig["direction"] == "down"].nlargest(n_label // 2, "score")])
    texts = [ax.text(r.log2fc, r.nlp, g, fontsize=7) for g, r in top.iterrows()]
    if HAS_ADJUSTTEXT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#999", lw=0.5))
    ax.annotate(f"↑ {(d['direction'] == 'up').sum():,} suben", xy=(0.98, 0.98), xycoords="axes fraction", ha="right", va="top", color=UP_COLOR, fontsize=10, fontweight="bold")
    ax.annotate(f"↓ {(d['direction'] == 'down').sum():,} bajan", xy=(0.02, 0.98), xycoords="axes fraction", ha="left", va="top", color=DOWN_COLOR, fontsize=10, fontweight="bold")
    ax.set_xlabel(xlabel); ax.set_ylabel("−log10 valor p ajustado"); ax.set_title(title)

## 4.1 · El diseño pareado, un gen cada vez

Cada donante se midió **con PBS y con cada citoquina**. Las dos mediciones de un donante *no son independientes*: comparten la genética, el sexo, la edad y el estado inmune basal del donante…
(La Parte 3 mostró fuertes efectos de donante.) Un **análisis pareado** compara cada donante **consigo mismo**, de modo que esas diferencias se cancelan. Con $y=\log_2(\text{CPM}+1)$:

$$d_{gi}\;=\;y^{\text{citoquina}}_{gi}-y^{\text{PBS}}_{gi}\qquad(g=\text{gen},\; i=\text{donante})$$

Ahora tenemos **12 números por gen** (uno por donante) y la pregunta pasa a ser: *¿es la media de estas 12 diferencias distinta de 0?*
Sigamos `EXAMPLE_GENE` en `CELL_TYPE`, desde los datos hasta el valor p.

In [ ]:
pbs_names = [f"{d}|{CELL_TYPE}|PBS" for d in DONORS]               # las 12 muestras control (una por donante)
cyt_names = [f"{d}|{CELL_TYPE}|{CYTOKINE}" for d in DONORS]        # las 12 muestras tratadas, en el MISMO orden de donantes

x_pbs = adata[pbs_names, EXAMPLE_GENE].X.ravel().astype(float)     # log2(CPM + 1) del gen en cada donante, con PBS…
x_cyt = adata[cyt_names, EXAMPLE_GENE].X.ravel().astype(float)     # …y con la citoquina
d     = x_cyt - x_pbs                                              # la diferencia pareada de cada donante

display(pd.DataFrame({"PBS": x_pbs, CYTOKINE: x_cyt, "diferencia d": d}, index=DONORS).round(2).T)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.2))
paired_plot(axes[0], x_pbs, x_cyt, EXAMPLE_GENE, CYTOKINE)
ctrl_pbs = adata[pbs_names, CONTROL_GENE].X.ravel().astype(float); ctrl_cyt = adata[cyt_names, CONTROL_GENE].X.ravel().astype(float)
paired_plot(axes[1], ctrl_pbs, ctrl_cyt, CONTROL_GENE, CYTOKINE)
fig.suptitle(f"{CT_LABEL[CELL_TYPE]}: cada línea es un donante (rojo: mayor con la citoquina, azul: menor)", y=1.02, fontsize=10)
plt.tight_layout(); plt.show()

**Izquierda:** *todos* los donantes suben con la citoquina, aunque las alturas de partida (los niveles basales) son muy distintas entre donantes: el análisis pareado mira el **cambio dentro de cada donante**.
**Derecha:** un gen constitutivo: las líneas van en ambos sentidos, sin cambio consistente.

### De 12 diferencias a un valor p

$$\log_2\text{FC}_g=\bar d_g=\frac1n\sum_i d_{gi},\qquad s_g=\sqrt{\frac{1}{n-1}\sum_i (d_{gi}-\bar d_g)^2},\qquad t_g=\frac{\bar d_g}{s_g/\sqrt n},\qquad p_g = 2\,\Pr\!\bigl(T_{n-1}>|t_g|\bigr)$$

Bajo la **hipótesis nula** (la citoquina no hace nada) $t_g$ sigue una distribución *t* de Student con $n-1=11$ grados de libertad. El **valor p** es la probabilidad de obtener un $|t|$ al menos tan grande como el nuestro *si la nula fuera cierta*.

In [ ]:
n      = len(d)                          # 12 donantes
mean_d = d.mean()                        # = el log2 fold change
sd_d   = d.std(ddof=1)                   # dispersión de las diferencias
se     = sd_d / np.sqrt(n)               # error estándar de la diferencia media
t_obs  = mean_d / se
p_val  = 2 * stats.t.sf(abs(t_obs), df=n - 1)

print(f"{EXAMPLE_GENE} en {CT_LABEL[CELL_TYPE]}, {CYTOKINE} vs PBS")
print(f"  diferencia media (log2FC) = {mean_d:.3f}   → fold change = {2 ** mean_d:.1f}×")
print(f"  DE de las diferencias     = {sd_d:.3f}")
print(f"  error estándar            = {sd_d:.3f} / √{n} = {se:.3f}")
print(f"  t = {mean_d:.3f} / {se:.3f} = {t_obs:.2f}        (grados de libertad = {n - 1})")
print(f"  p = 2 · Pr(T > {abs(t_obs):.2f}) = {p_val:.1e}")
ref = stats.ttest_rel(x_cyt, x_pbs)
print(f"\nscipy.stats.ttest_rel da t = {ref.statistic:.2f}, p = {ref.pvalue:.1e}   (idéntico ✔)")

xx = np.linspace(-max(6, abs(t_obs) * 1.25), max(6, abs(t_obs) * 1.25), 500)
fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(xx, stats.t.pdf(xx, df=n - 1), color="k", lw=1.5)
for sgn in (1, -1):
    tail = xx[sgn * xx >= abs(t_obs)]
    ax.fill_between(tail, stats.t.pdf(tail, df=n - 1), color=UP_COLOR, alpha=0.7)
ax.axvline(t_obs, color=UP_COLOR, ls="--")
ax.annotate(f"t observado = {t_obs:.1f}\np = {p_val:.1e}", xy=(t_obs, 0.2), xytext=(-8, 0), textcoords="offset points", ha="right", color=UP_COLOR, fontsize=9)
ax.set_xlabel("estadístico t"); ax.set_ylabel("densidad"); ax.set_title(f"distribución t con {n - 1} grados de libertad (hipótesis nula: sin efecto)")
plt.tight_layout(); plt.show()

Las colas sombreadas son la probabilidad de ver algo *al menos así de extremo* por azar: diminuta, así que los datos son difíciles de conciliar con "sin efecto".

> 🔧 **Te toca:** cambia `EXAMPLE_GENE` (p. ej. a `"IDO1"`, `"STAT1"` o `"ACTB"`) en la celda de *parámetros* y vuelve a ejecutar las tres celdas anteriores. ¿Cuál obtiene el **menor valor p**: el gen con mayor fold change o el gen cuyas diferencias son más **consistentes** entre donantes?

## 4.2 · La misma pregunta para todos los genes: DESeq2

Una prueba t sobre log-CPM es una buena herramienta didáctica, pero los estudios reales usan herramientas diseñadas para **conteos**. La más popular es **DESeq2** (R); `pydeseq2` es su implementación en Python. A grandes rasgos hace cuatro cosas:

| Paso | Qué hace DESeq2 | Por qué |
|:--:|---|---|
| **1. Factores de tamaño** | Trabaja sobre los **conteos crudos** y reescala cada muestra con el método de la *mediana de cocientes*: el factor de tamaño de una muestra es la mediana, entre genes, de su cociente respecto a la media del gen en todas las muestras | Una versión robusta de los CPM: asume que *la mayoría de genes no cambia*, así que unos pocos genes muy inducidos no pueden distorsionar la escala (el efecto de composición de la Parte 2) |
| **2. Modelo** | Ajusta para cada gen un modelo lineal generalizado **binomial negativo**: $\log \mu_{gi}=\log s_i+\beta_{g,\text{donante}(i)}+\beta_{g,\text{citoquina}(i)}$ | Los conteos no se distribuyen normalmente: la varianza crece con la media. `~ donor + cytokine` es el **diseño pareado**: cada donante tiene su propio nivel basal y `cytokine` es el efecto que queremos |
| **3. Dispersión** | Estima el ruido de cada gen y lo **contrae hacia una tendencia** común a todos los genes | Con solo 12 donantes, la estimación del ruido de un gen es poco fiable; compartir información entre genes hace la prueba más estable |
| **4. Prueba** | Una **prueba de Wald** del coeficiente de la citoquina: $\log_2\text{FC}$, su error estándar, un valor p | La misma lógica que la prueba t: *¿es el efecto mayor que el ruido?* |

Después corregimos los valores p por los ~13,000 genes que probamos (sección siguiente). La función de abajo ejecuta todo para una citoquina frente a otra condición en un tipo celular. Aplica **el mismo filtro de expresión** de la Parte 2 a las 24 muestras de la comparación.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def run_deseq2(adata, cell_type, treated, reference, padj_thresh=PADJ_THRESH, lfc_thresh=LFC_THRESH):
    '''DESeq2 (pydeseq2): `treated` vs `reference` en un tipo celular, 12 donantes, diseño "~ donor + cytokine". Una fila por gen.'''
    names = [f"{d}|{cell_type}|{c}" for c in (reference, treated) for d in DONORS]         # 12 muestras de referencia + 12 tratadas
    sub   = adata[names]
    counts = pd.DataFrame(sub.layers["counts"].astype(int), index=names, columns=adata.var_names)     # conteos CRUDOS, enteros
    counts = counts.loc[:, is_expressed(counts.values)]                                    # filtro de expresión (Parte 2) sobre estas 24 muestras
    meta = pd.DataFrame({"donor": DONORS * 2, "cytokine": [reference] * 12 + [treated] * 12}, index=names)

    dds = DeseqDataSet(counts=counts, metadata=meta, design="~ donor + cytokine", quiet=True)
    dds.deseq2()                                                                           # factores de tamaño → dispersiones → ajuste del modelo
    stat = DeseqStats(dds, contrast=["cytokine", treated, reference], quiet=True)          # prueba de Wald de "treated vs reference"
    stat.summary()                                                                         # valores p y valores p ajustados de Benjamini–Hochberg

    res = stat.results_df.rename(columns={"log2FoldChange": "log2fc", "pvalue": "pval", "baseMean": "base_mean"}).reindex(adata.var_names)
    for col, name in [("genewise_dispersions", "disp_genewise"), ("fitted_dispersions", "disp_trend"), ("dispersions", "disp_final")]:
        res[name] = dds.var[col].reindex(adata.var_names)                                  # los tres pasos de la estimación de la dispersión
    res["tested"] = res["padj"].notna()
    res["direction"] = "ns"
    res.loc[res["tested"] & (res["padj"] < padj_thresh) & (res["log2fc"] >  lfc_thresh), "direction"] = "up"
    res.loc[res["tested"] & (res["padj"] < padj_thresh) & (res["log2fc"] < -lfc_thresh), "direction"] = "down"
    res.attrs["size_factors"] = dds.obs["size_factors"].to_dict()                          # guardados para verlos más abajo
    res.attrs["n_filtered_in"] = counts.shape[1]
    return res

de = run_deseq2(adata, CELL_TYPE, CYTOKINE, "PBS")          # ~ 3 segundos

print(f"{CYTOKINE} vs PBS en {CT_LABEL[CELL_TYPE]}")
print(f"  genes en el objeto         : {len(de):,}")
print(f"  superan el filtro          : {de.attrs['n_filtered_in']:,}")
print(f"  analizados (con p ajust.)  : {de['tested'].sum():,}")
print(f"  sobreexpresados (DEG)      : {(de['direction'] == 'up').sum():,}")
print(f"  infraexpresados (DEG)      : {(de['direction'] == 'down').sum():,}")
with pd.option_context("display.float_format", "{:.3g}".format):
    print("\nNuestros dos genes de ejemplo, en la tabla de resultados:"); display(de.loc[[EXAMPLE_GENE, CONTROL_GENE]])
    print("Los 8 genes con menor valor p:");        display(de[de["tested"]].sort_values("pval").head(8))

| Columna | Significado |
|---|---|
| `base_mean` | media de los conteos *normalizados* en las 24 muestras |
| `log2fc` | $\log_2$ fold change, `treated` vs `reference` (+1 = se duplica) |
| `lfcSE`, `stat`, `pval` | error estándar, estadístico de Wald y su valor p: **cuánto sorprende** el cambio si no pasa nada |
| `padj` | valor p **ajustado por pruebas múltiples** (Benjamini–Hochberg): ver la sección siguiente |
| `tested` | el gen tiene un valor p ajustado (pasó los filtros) |
| `direction` | `up` / `down` si `padj < PADJ_THRESH` **y** `|log2fc| > LFC_THRESH`, en caso contrario `ns` (no significativo) |

**¿Cuánto se diferencia de nuestra prueba t sobre log-CPM?** Ejecutemos ambas con todos los genes y comparemos:

In [ ]:
def paired_ttest_de(adata, cell_type, treated, reference):
    '''La prueba t pareada de 4.1 sobre log2(CPM + 1), para todos los genes a la vez (+ corrección BH).'''
    ir = adata.obs_names.get_indexer([f"{d}|{cell_type}|{reference}" for d in DONORS])
    it = adata.obs_names.get_indexer([f"{d}|{cell_type}|{treated}" for d in DONORS])
    diff   = adata.X[it].astype(np.float64) - adata.X[ir]                                   # 12 donantes × genes
    log2fc = diff.mean(axis=0)
    se     = diff.std(axis=0, ddof=1) / np.sqrt(len(DONORS))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = log2fc / se
    pval   = np.where(se > 0, 2 * stats.t.sf(np.abs(t), df=len(DONORS) - 1), 1.0)
    tested = is_expressed(np.vstack([adata.layers["counts"][ir], adata.layers["counts"][it]]))
    padj   = np.full(len(pval), np.nan); padj[tested] = multipletests(pval[tested], method="fdr_bh")[1]
    return pd.DataFrame({"log2fc": log2fc, "pval": pval, "padj": padj, "tested": tested}, index=adata.var_names)

de_t = paired_ttest_de(adata, CELL_TYPE, CYTOKINE, "PBS")
both = de["tested"] & de_t["tested"]

sig_d = (de["padj"] < PADJ_THRESH) & (de["log2fc"].abs() > LFC_THRESH) & de["tested"]
sig_t = (de_t["padj"] < PADJ_THRESH) & (de_t["log2fc"].abs() > LFC_THRESH) & de_t["tested"]
print(f"DEG: DESeq2 {sig_d.sum():,} · prueba t {sig_t.sum():,} · ambos {(sig_d & sig_t).sum():,}")
print(f"Mediana del log2FC de los genes analizados: DESeq2 {de.loc[de['tested'], 'log2fc'].median():+.3f} · prueba t sobre log-CPM {de_t.loc[de_t['tested'], 'log2fc'].median():+.3f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
ax = axes[0]
ax.scatter(de_t.loc[both, "log2fc"], de.loc[both, "log2fc"], s=3, alpha=0.3, color="#2a78d6", linewidths=0, rasterized=True)
lim = 4.5; ax.plot([-lim, lim], [-lim, lim], color="k", lw=0.8, ls="--"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
r = np.corrcoef(de_t.loc[both, "log2fc"], de.loc[both, "log2fc"])[0, 1]
ax.set_xlabel("log2FC, diferencia log-CPM"); ax.set_ylabel("log2FC, DESeq2"); ax.set_title(f"Tamaños de efecto (r = {r:.2f})")

# factores de tamaño frente al tamaño de librería: la normalización de DESeq2 es una versión robusta de los CPM
sf = pd.Series(de.attrs["size_factors"]); libsz = adata.obs.loc[sf.index, "lib_size"].values
ax = axes[1]
is_ref = np.array([n.endswith("|PBS") for n in sf.index])
ax.scatter(libsz[is_ref], sf.values[is_ref], s=40, color=PBS_COLOR, label="PBS"); ax.scatter(libsz[~is_ref], sf.values[~is_ref], s=40, color=CYTOKINE_COLOR, label=CYTOKINE)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("tamaño de librería"); ax.set_ylabel("factor de tamaño de DESeq2"); ax.legend(loc="upper left")
ax.set_title(f"Los factores de tamaño siguen al tamaño de librería (r = {np.corrcoef(np.log(libsz), np.log(sf.values))[0, 1]:.3f})")

# dispersión: el ruido de cada gen. Estimación cruda (gris) → contraída hacia la tendencia (línea roja) → valor final (azul)
dd = de[de["tested"] & (de["disp_genewise"] > 1e-6)]
ax = axes[2]
ax.scatter(dd["base_mean"], dd["disp_genewise"], s=3, alpha=0.4, color=PBS_COLOR, linewidths=0, rasterized=True)
ax.scatter(dd["base_mean"], dd["disp_final"], s=3, alpha=0.25, color="#2a78d6", linewidths=0, rasterized=True)
o_ = np.argsort(dd["base_mean"].values); ax.plot(dd["base_mean"].values[o_], dd["disp_trend"].values[o_], color=UP_COLOR, lw=2)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("media de conteos normalizados"); ax.set_ylabel("dispersión (ruido)"); ax.set_title("Dispersión: contraída hacia la tendencia")
ax.legend(handles=[Line2D([], [], marker="o", color=PBS_COLOR, ls="", label="estimación por gen"), Line2D([], [], marker="o", color="#2a78d6", ls="", label="final (contraída)"),
                   Line2D([], [], color=UP_COLOR, lw=2, label="tendencia")], loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

**❓ Preguntas**
1. Los dos enfoques dan tamaños de efecto parecidos. ¿Dónde difieren más? Mira la **mediana del log₂FC** impresa arriba: ¿cuál está más cerca de 0?
2. ¿Por qué los factores de tamaño son casi proporcionales al tamaño de librería (panel central)? ¿Qué tienen de distinto?
3. Panel derecho: ¿qué pasa con las ruidosas estimaciones de dispersión por gen (gris) cuando DESeq2 las contrae hacia la tendencia? ¿Por qué ayuda con 12 donantes?

<details>
<summary>💡 Respuestas</summary>

1. Los tamaños de los efectos coinciden bien, pero la **mediana del log₂FC de DESeq2 está más cerca de 0** que la de log-CPM (−0.09): la mediana de cocientes asume que la mayoría de genes no cambia, así que los genes muy inducidos no arrastran hacia abajo al gen típico (el *efecto de composición* de la Parte 2, extra B2). DESeq2 también da más peso a genes con muchos conteos y menos a los ruidosos con pocos conteos.
2. Porque el tamaño de librería *es* la principal diferencia técnica entre muestras; los factores de tamaño son una versión robusta de él, estimada a partir de los genes que no cambian en lugar del total.
3. La nube gris es ancha (con 12 muestras la estimación propia de cada gen es ruidosa); los puntos azules se acercan a la tendencia roja, así que los valores extremos se moderan. Menos genes parecen "demasiado ruidosos" o "demasiado limpios" por azar, y los valores p son más fiables.

</details>

## 4.3 · Pruebas múltiples: por qué corregimos los valores p

Cada gen se prueba al nivel del 5 %: *si no pasara absolutamente nada*, el 5 % de los genes obtendría igualmente $p<0.05$ por azar. Con ~13,000 genes probados son **~600 falsos positivos**. La **Tasa de Falsos Descubrimientos (FDR)** es la fracción esperada de falsos positivos *entre los genes que llamamos significativos*.

In [ ]:
tested = de["tested"].values
n_tested = tested.sum()
print(f"Genes analizados: {n_tested:,}   →   falsos positivos esperados con p < 0.05 si NO pasara NADA: {0.05 * n_tested:,.0f}")
print(f"Genes con p cruda < 0.05           : {(de.loc[tested, 'pval'] < 0.05).sum():,}")
print(f"Genes con p ajustada (padj) < 0.05 : {(de.loc[tested, 'padj'] < 0.05).sum():,}")

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(de.loc[tested, "pval"], bins=40, range=(0, 1), color="#2a78d6")
ax.axhline(n_tested / 40, color="k", ls="--", lw=0.8); ax.annotate("lo que veríamos si no pasara nada", xy=(0.97, n_tested / 40), xytext=(0, 4), textcoords="offset points", ha="right", fontsize=8)
ax.set_xlabel("valor p"); ax.set_ylabel("número de genes"); ax.set_title(f"{CYTOKINE} vs PBS · {CT_LABEL[CELL_TYPE]}")
plt.tight_layout(); plt.show()

El suelo plano es el "ruido" (la línea discontinua); el **pico cerca de 0** son los genes que de verdad responden.

**Benjamini–Hochberg (BH)** controla la FDR: entre los genes que llamamos significativos con `padj < 0.05`, *esperamos* que como máximo el 5 % sean falsos positivos.
Con $m$ pruebas y valores p ordenados $p_{(1)}\le\dots\le p_{(m)}$:

$$p^{\text{adj}}_{(i)}\;=\;\min_{k\ge i}\Bigl(\tfrac{m}{k}\,p_{(k)}\Bigr)\quad(\text{con tope en }1)$$

es decir, *cada valor p se multiplica por $m$ y se divide por su rango* (y luego se hace monótono). Un ejemplo diminuto con 10 genes:

In [ ]:
p_toy = np.array([0.001, 0.008, 0.012, 0.030, 0.040, 0.045, 0.200, 0.400, 0.700, 0.900])      # 10 valores p, ordenados
m, rank = len(p_toy), np.arange(1, len(p_toy) + 1)
step1 = p_toy * m / rank                                           # p × m / rango
bh    = np.minimum(np.minimum.accumulate(step1[::-1])[::-1], 1)    # hacerlo monótono, desde el rango mayor hacia abajo
display(pd.DataFrame({"valor p": p_toy, "rango": rank, "p × m / rango": step1.round(3), "BH ajustado": bh.round(3),
                      "statsmodels": multipletests(p_toy, method="fdr_bh")[1].round(3)}, index=[f"gen {i}" for i in rank]))
print(f"p cruda < 0.05: {(p_toy < 0.05).sum()} genes   →   p ajustada < 0.05: {(bh < 0.05).sum()} genes")

Seis genes pasan el umbral crudo de 0.05, pero solo tres sobreviven tras la corrección: los demás *no eran lo bastante fuertes dado cuántos genes probamos*.

## 4.4 · ¿Qué llamamos "diferencialmente expresado"? El gráfico de volcán

Un valor p pequeño dice que el efecto **probablemente no es cero**, no que sea **grande**. Con suficientes donantes incluso un cambio del 1 % obtiene un valor p diminuto, pero puede ser biológicamente irrelevante. Por eso pedimos **ambas cosas**:

$$\text{DEG}\;\Longleftrightarrow\;\texttt{padj}<\texttt{PADJ\_THRESH}\;\;\text{y}\;\;|\log_2\text{FC}|>\texttt{LFC\_THRESH}$$

El **gráfico de volcán** muestra ambas a la vez: *x* = log₂FC (tamaño del efecto), *y* = −log₁₀ valor p ajustado (evidencia).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
volcano(ax, de, f"{CYTOKINE} vs PBS · {CT_LABEL[CELL_TYPE]}")
plt.tight_layout(); plt.show()

**❓ Preguntas**
1. ¿Qué genes reconoces entre las etiquetas (los cambios más fuertes)? Ahora busca `IDO1`, `GBP1` y `CXCL10` en la tabla `de`: ¿son DEG? ¿Qué papel biológico tienen estos genes?
2. Genes en el **centro superior** frente a genes en los **lados pero abajo**: ¿qué significa cada caso?
3. ¿Cuántos genes se llamarían DEG con `LFC_THRESH = 0`, y con `1`? (cámbialo en la celda de parámetros y vuelve a ejecutar desde 4.2). ¿Qué aporta el umbral de fold change?

<details>
<summary>💡 Respuestas</summary>

1. Las etiquetas son los DEG con mejor combinación de evidencia y tamaño de efecto. En monocitos con IFN-γ, entre los sobreexpresados deberías encontrar `IL32` y `UBD`, y entre los infraexpresados los genes de monocito/macrófago `MRC1`, `CD14` y `VSIG4`: IFN-γ aleja a las células de su estado de reposo. `de.loc[["CXCL9", "CXCL10", "IDO1", "GBP1"]]` muestra que las quimioquinas atrayentes de linfocitos T `CXCL9` y `CXCL10`, la enzima que degrada triptófano `IDO1` y la proteína de unión a guanilato `GBP1` también son DEG (suben ×3 a ×14): la clásica **respuesta a interferón-γ**, es decir, reclutamiento de linfocitos T y programas antimicrobianos.
2. **Centro superior**: muy significativo pero con un efecto *diminuto* (poco ruido, poca diferencia). **Lados pero abajo**: grandes cambios medios *no consistentes entre donantes* (no significativos).
3. `LFC_THRESH = 0` conserva muchos más genes (todos los significativos, incluidos cambios muy pequeños) y `1` solo los más fuertes. El umbral de fold change elimina cambios estadísticamente significativos pero biológicamente despreciables.

</details>

## 4.5 · Comparar dos citoquinas

Hasta ahora comparamos una citoquina con PBS. Ahora comparamos **dos citoquinas**: ¿qué genes *comparten* y cuáles son **específicos** de cada una? Para cada tipo celular de `CELL_TYPES_COMPARE` ejecutamos DESeq2 tres veces (los gráficos y la tabla solo usan los genes probados en *ambas* comparaciones):

* `CYTOKINE_A` vs PBS y `CYTOKINE_B` vs PBS, y ponemos los dos fold changes uno junto al otro;
* `CYTOKINE_A` vs `CYTOKINE_B` **directamente**: los genes que difieren entre las dos citoquinas.

In [ ]:
# 🔧 CAMBIA ESTO (en la celda de parámetros de arriba): CYTOKINE_A, CYTOKINE_B y CELL_TYPES_COMPARE (1 o 2 tipos celulares). ~ 7 segundos por tipo celular
cmp = {}
for ct in CELL_TYPES_COMPARE:
    cmp[ct] = dict(A=run_deseq2(adata, ct, CYTOKINE_A, "PBS"), B=run_deseq2(adata, ct, CYTOKINE_B, "PBS"), AB=run_deseq2(adata, ct, CYTOKINE_A, CYTOKINE_B))

COL_A, COL_B, COL_BOTH = "#2a78d6", "#eb6834", "#6250d6"
fig, axes = plt.subplots(len(CELL_TYPES_COMPARE), 2, figsize=(12, 5.2 * len(CELL_TYPES_COMPARE)), squeeze=False)
rows = []
for r_, ct in enumerate(CELL_TYPES_COMPARE):
    A, B, AB = cmp[ct]["A"], cmp[ct]["B"], cmp[ct]["AB"]
    ok = A["tested"] & B["tested"]                                      # solo se comparan los genes analizados en ambas comparaciones
    sig_a, sig_b = (A["direction"] != "ns") & ok, (B["direction"] != "ns") & ok
    ax = axes[r_, 0]
    for mask_, col, lab in [(ok & ~sig_a & ~sig_b, NS_COLOR, "ninguna"), (ok & sig_a & ~sig_b, COL_A, f"solo {CYTOKINE_A}"),
                            (ok & ~sig_a & sig_b, COL_B, f"solo {CYTOKINE_B}"), (ok & sig_a & sig_b, COL_BOTH, "ambas")]:
        ax.scatter(A.loc[mask_, "log2fc"], B.loc[mask_, "log2fc"], s=5, alpha=0.55, color=col, linewidths=0, label=f"{lab} ({mask_.sum():,})", rasterized=True)
    ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(f"log2FC  {CYTOKINE_A} vs PBS"); ax.set_ylabel(f"log2FC  {CYTOKINE_B} vs PBS"); ax.set_title(f"{CT_LABEL[ct]}: respuestas compartidas y específicas")
    ax.legend(loc="upper left", markerscale=3, fontsize=8)
    volcano(axes[r_, 1], AB, f"{CT_LABEL[ct]}: {CYTOKINE_A} vs {CYTOKINE_B}", n_label=10, xlabel=f"log2 fold change ({CYTOKINE_A} / {CYTOKINE_B})")
    rows.append({"tipo celular": CT_LABEL[ct], f"DEG {CYTOKINE_A}": int(sig_a.sum()), f"DEG {CYTOKINE_B}": int(sig_b.sum()),
                 "compartidos": int((sig_a & sig_b).sum()), f"solo {CYTOKINE_A}": int((sig_a & ~sig_b).sum()), f"solo {CYTOKINE_B}": int((~sig_a & sig_b).sum()),
                 f"DEG {CYTOKINE_A} vs {CYTOKINE_B}": int((AB["direction"] != "ns").sum())})
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index("tipo celular"))

**❓ Preguntas**
1. Mira los gráficos de la izquierda. Los genes a lo largo de la diagonal responden *igual* a ambas citoquinas; los genes cerca de un eje responden solo a una (colores: azul = solo A, naranja = solo B, violeta = ambas). ¿Qué ves para IFN-γ e IL-4? ¿Están todos los genes "ambas" en la diagonal?
2. ¿Actúan las dos citoquinas de forma más distinta en monocitos o en linfocitos T CD4? ¿Cómo lo sabes a partir de la tabla?
3. El **volcán de la derecha** compara directamente las dos citoquinas. ¿Qué genes son mayores con IFN-γ que con IL-4? ¿Coinciden con los genes *solo IFN-γ* de la izquierda?

<details>
<summary>💡 Respuestas</summary>

1. La nube **no** se concentra en la diagonal: la mayoría de los DEG son específicos de una citoquina, y el número de genes llamados DEG en ambas es menor que la suma de los específicos (mira la leyenda y la tabla). Incluso entre los genes "ambas" (violeta) muchos van en **sentidos opuestos** (cuadrante inferior derecho: suben con IFN-γ, bajan con IL-4), lo que encaja con que los dos programas sean distintos (Parte 2, mapa de calor de genes inmunes).
2. En **monocitos**: tienen muchos más DEG y también muchos más genes que difieren entre las dos citoquinas (última columna). Los linfocitos T responden menos a ambas.
3. Revisa las etiquetas del volcán: en monocitos, los genes más fuertes a la derecha (mayores con IFN-γ) incluyen `ACOD1`, un marcador de células mieloides activadas por IFN-γ, y a la izquierda (mayores con IL-4) `ALOX15`, una diana clásica de IL-4. Muchos de ellos están entre los genes "solo A" y "solo B" del gráfico izquierdo, porque un gen que responde a una citoquina y no a la otra difiere entre ellas.

</details>

## 4.6 · 🔧 Te toca

1. En la celda de **parámetros**, cambia `CYTOKINE` a `"IL-4"` y `EXAMPLE_GENE` a `"ALOX15"`, y vuelve a ejecutar desde 4.1. ¿Qué conclusiones siguen siendo válidas?
2. Elige **dos citoquinas cualesquiera** de las 87 (`CYTOKINE_A`, `CYTOKINE_B`), por ejemplo dos interferones (`"IFN-alpha1"` e `"IFN-gamma"`) o dos citoquinas de la misma familia, y uno o dos tipos celulares. ¿Qué pareja es la más parecida? ¿La más distinta?
3. Cambia `PADJ_THRESH` a `0.01` y `LFC_THRESH` a `1`. ¿Cómo cambia el número de DEG?

## 4.7 · Mensajes para llevarse

| | |
|---|---|
| **Diseño** | Pseudobulk: la suma de las células de un donante. La unidad de replicación es el **donante**, no la célula. `~ donor + cytokine` es un diseño pareado: elimina los efectos de donante compartidos. |
| **Herramienta** | **DESeq2** trabaja con conteos crudos (factores de tamaño por mediana de cocientes, modelo binomial negativo, dispersiones contraídas) y es el estándar en estudios reales. `limma-voom` y `edgeR` son alternativas. |
| **Pruebas múltiples** | ~13,000 pruebas: informa siempre de valores p **ajustados** (FDR, Benjamini–Hochberg). |
| **Significación ≠ importancia** | Combina `padj` con un umbral de tamaño de efecto y mira los donantes individuales. |
| **Filtrado** | Elimina los genes no analizables *antes* de la corrección. |
| **Siguiente paso** | Una lista de DEG se convierte en biología mediante el **enriquecimiento funcional** (vías, dianas de factores de transcripción). |

---
# 🏁 Cierre

Has pasado de un solo número a un resultado de expresión diferencial:

**conteos crudos** → `AnnData` → **tamaños de librería** → **CPM y log₂** → **filtro de genes** → **PCA** (¿qué varía?) → **DESeq2 + FDR** (¿qué cambia?) → **dos citoquinas** (¿qué se comparte y qué es específico?)

**¿Y ahora qué?**

* **Enriquecimiento funcional** de las listas de DEG (vías, dianas de factores de transcripción): `gseapy`, `decoupler`.
* **Análisis de célula única** de los datos originales: `scanpy`; buenas prácticas en Heumos *et al.* (2023), *Nat Rev Genet*.
* **El propio Diccionario Humano de Citoquinas** y su paquete `huCIRA` (Oesinghaus *et al.*, 2025) para interpretar tus propios datos con firmas de respuesta a citoquinas.

**Referencias**
* Oesinghaus L. *et al.* (2025) A single-cell cytokine dictionary of human peripheral blood. *bioRxiv*, doi:10.64898/2025.12.12.693897
* Virshup I. *et al.* (2024) anndata: Access and store annotated data matrices. *J. Open Source Softw.* 9:4371
* Heumos L. *et al.* (2023) Best practices for single-cell analysis across modalities. *Nat. Rev. Genet.* 24:550–572
* Love M.I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome Biol.* 15:550
* Muzellec B. *et al.* (2023) PyDESeq2: a python package for bulk RNA-seq differential expression analysis. *Bioinformatics* 39:btad547
* Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate. *J. R. Stat. Soc. B* 57:289–300

---
# 🎁 Extras: solo si te sobra tiempo (o en casa)

## B1 · El rompecabezas del PBS: comprueba siempre cómo se construyó tu control

En 2.1 descartamos 6 de las 7 filas de PBS de cada donante y tipo celular. *¿Por qué era correcto?* Miremos el archivo crudo.

In [ ]:
import scipy.sparse as sp

raw_obs = adata_raw.obs
ex = raw_obs[(raw_obs["cytokine"] == "PBS") & (raw_obs["donor"] == "Donor1") & (raw_obs["cell_type"] == "CD14_Mono")]
print("Las 7 filas de PBS de Donor1 / CD14_Mono:")
display(ex[["sample", "bc1_well", "psbulk_n_cells"]])
print(f"6 pocillos individuales (H7–H12) → {ex.loc[ex['bc1_well'].notna(), 'psbulk_n_cells'].sum():,.0f} células en total;   la fila sin pocillo tiene {ex.loc[ex['bc1_well'].isna(), 'psbulk_n_cells'].iloc[0]:,.0f} células")

La 7.ª fila parece la **suma de los seis pocillos**. *Verifiquémoslo* en todos los grupos donante × tipo celular en lugar de fiarnos:

In [ ]:
is_pbs, is_well = (raw_obs["cytokine"] == "PBS").values, raw_obs["bc1_well"].notna().values
Xr  = sp.csr_matrix(adata_raw.X)
pos = pd.Series(np.arange(len(raw_obs)), index=raw_obs.index)

n_groups, all_equal = 0, True
for (ct, donor), grp in raw_obs[is_pbs & is_well].groupby(["cell_type", "donor"], observed=True):
    sum_of_wells = np.asarray(Xr[pos[grp.index].values].sum(axis=0)).ravel()
    total_name   = raw_obs[is_pbs & ~is_well & (raw_obs["cell_type"] == ct) & (raw_obs["donor"] == donor)].index[0]
    all_equal   &= np.allclose(sum_of_wells, Xr[pos[total_name]].toarray().ravel())
    n_groups    += 1
print(f"Comprobados {n_groups} grupos → la suma de los 6 pocillos es igual a la fila de PBS ya calculada en todos ellos: {all_equal}")

✅ Así que conservamos la fila total y descartamos los pocillos: **un perfil de PBS por donante y tipo celular**, como cualquier citoquina.

> ⚠️ **Trampa frecuente:** si sumaras *las siete* filas contarías cada célula de PBS dos veces. Tras los CPM el resultado parecería el mismo, pero el número de células sería incorrecto: justo el tipo de error silencioso que hace irreproducibles los análisis. *Comprueba, no asumas.*

## B2 · Lo que los CPM no pueden arreglar: efectos de composición

Los CPM son una **proporción**: si un gen toma una parte mayor, *todos los demás* deben tomar una menor. Las dos muestras de abajo tienen la misma profundidad (10,000 conteos) y **solo el gen E está realmente inducido** (6×):

In [ ]:
true = pd.DataFrame({"control": [2000, 2000, 2000, 2000, 2000], "treated": [2000, 2000, 2000, 2000, 12000]}, index=list("ABCDE"))
seq  = (true / true.sum() * 10_000).round()                      # lo que observamos con una profundidad de secuenciación fija de 10,000
seq["log2FC observado"] = np.log2(seq["treated"] / seq["control"]).round(2)
seq["realidad"] = ["sin cambio"] * 4 + ["↑ 6×"]
display(seq)

Los genes A–D **no** cambiaron, y sin embargo parecen 2× menores porque el gen E ahora ocupa una gran parte de la muestra: un **efecto de composición**. Es la razón de métodos como *TMM* y la *mediana de cocientes de DESeq2*:
asumen que **la mayoría de genes no cambia** y reescalan en consecuencia. Compara la distribución del log₂FC de todos los genes con las dos normalizaciones (la comparación de 4.2):

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.linspace(-3, 3, 90)
for res_, col, lab in [(de_t, PBS_COLOR, "diferencia log-CPM"), (de, "#2a78d6", "DESeq2")]:
    v = res_.loc[res_["tested"], "log2fc"]
    ax.hist(v, bins=bins, histtype="stepfilled", alpha=0.55, color=col, label=f"{lab} (mediana {v.median():+.2f})")
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("log2 fold change"); ax.set_ylabel("número de genes"); ax.legend(loc="upper left")
ax.set_title(f"{CYTOKINE} vs PBS · {CT_LABEL[CELL_TYPE]}")
plt.tight_layout(); plt.show()

Con CPM toda la distribución se desplaza ligeramente por debajo de 0 (unos pocos genes muy inducidos ocupan parte de la librería); los factores de tamaño de DESeq2 están centrados. En un estudio real, usa TMM / mediana de cocientes, como hace DESeq2.

## B3 · Un experimento nulo: ¿cuántos falsos positivos obtenemos realmente?

Dejamos los datos "libres de citoquina": **(i)** quitamos el efecto medio de cada gen y **(ii)** invertimos al azar el signo de las diferencias restantes de cada donante (si la citoquina no hiciera nada, "citoquina − PBS" y "PBS − citoquina" serían igual de probables).
El ruido es el ruido real, el efecto es cero, y ejecutamos exactamente la misma prueba t pareada:

In [ ]:
def paired_pvalues(diff):
    n  = diff.shape[0]
    se = diff.std(axis=0, ddof=1) / np.sqrt(n)
    with np.errstate(divide="ignore", invalid="ignore"):
        t = diff.mean(axis=0) / se
    return np.where(se > 0, 2 * stats.t.sf(np.abs(t), df=n - 1), 1.0)

ip = adata.obs_names.get_indexer(pbs_names); ic = adata.obs_names.get_indexer(cyt_names)
diff  = (adata.X[ic] - adata.X[ip]).astype(np.float64)
noise = diff - diff.mean(axis=0)                                                    # (i) el mismo ruido, sin efecto
flip  = np.random.default_rng(1).choice([-1.0, 1.0], size=(len(DONORS), 1))         # (ii) un signo aleatorio por donante
tst   = de_t["tested"].values
p_null = paired_pvalues(noise * flip)[tst]
p_real = de_t.loc[tst, "pval"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, p, title, col in [(axes[0], p_real, "Comparación REAL (prueba t)", "#2a78d6"), (axes[1], p_null, "NULA: sin efecto, mismo ruido", PBS_COLOR)]:
    ax.hist(p, bins=40, range=(0, 1), color=col); ax.axhline(len(p) / 40, color="k", ls="--", lw=0.8)
    ax.set_xlabel("valor p"); ax.set_title(f"{title}\n{(p < 0.05).sum():,} genes con p < 0.05")
axes[0].set_ylabel("número de genes")
plt.tight_layout(); plt.show()

padj_null = multipletests(p_null, method="fdr_bh")[1]
print(f"Datos NULOS: {(p_null < 0.05).sum():,} genes con p cruda < 0.05 (5 % de {len(p_null):,} = {0.05 * len(p_null):.0f} esperados)")
print(f"Datos NULOS: {(padj_null < 0.05).sum():,} genes con p ajustada < 0.05   ← tras Benjamini–Hochberg")

En los datos nulos **~5 % de los genes** tienen $p<0.05$ por puro azar, y los valores p son uniformes (un histograma plano). Tras la corrección, prácticamente **ninguno** se llama significativo. Eso es lo que aporta el control de la FDR.

## B4 · ¿Ayuda realmente el pareado?

El pareado elimina el nivel basal del donante **solo si ese nivel basal es compartido por las dos condiciones**, es decir, si los valores de PBS y de citoquina están **correlacionados entre donantes**. Compara la prueba pareada con una **no pareada** (Welch), que ignora qué muestras proceden del mismo donante:

In [ ]:
rows = []
for g in dict.fromkeys([EXAMPLE_GENE, "ISG15", "IFI6", "MX1", "STAT1", CONTROL_GENE]):
    a0 = adata[pbs_names, g].X.ravel().astype(float); a1 = adata[cyt_names, g].X.ravel().astype(float)
    rows.append({"gene": g, "log2FC": (a1 - a0).mean(), "r (PBS vs citoquina, entre donantes)": np.corrcoef(a0, a1)[0, 1],
                 "p pareada": stats.ttest_rel(a1, a0).pvalue, "p no pareada": stats.ttest_ind(a1, a0, equal_var=False).pvalue})
with pd.option_context("display.float_format", "{:.3g}".format):
    display(pd.DataFrame(rows).set_index("gene"))

En los **genes estimulados por interferón** (`ISG15`, `IFI6`, `MX1`) los niveles basales de los donantes se comparten mucho: algunos donantes tienen alta expresión de ISG *con y sin* la citoquina.
La prueba no pareada ahoga el efecto en la dispersión entre donantes, y la **prueba pareada es mucho más sensible**. Donde los valores de PBS y de citoquina *no* están correlacionados positivamente, el pareado no aporta nada (y cuesta grados de libertad).
En cualquier caso el experimento **es** pareado, así que el diseño pareado (`~ donor + cytokine` en DESeq2) es el que le corresponde.

## B5 · Exploración libre

Elige **cualquier citoquina** de las 87 y compara los cuatro tipos celulares (unos 12 segundos).

In [ ]:
# 🔧 CAMBIA ESTO: cualquier citoquina de adata.obs["cytokine"]
EXPLORE_CYTOKINE = "IL-15"

if EXPLORE_CYTOKINE not in adata.obs["cytokine"].cat.categories:
    print(f"⚠️  '{EXPLORE_CYTOKINE}' no existe. Las citoquinas son:\n", list(adata.obs["cytokine"].cat.categories))
else:
    de_explore = {ct: run_deseq2(adata, ct, EXPLORE_CYTOKINE, "PBS") for ct in CELL_TYPES}
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
    for ax, ct in zip(axes.ravel(), CELL_TYPES):
        volcano(ax, de_explore[ct], CT_LABEL[ct], n_label=10)
    fig.suptitle(f"{EXPLORE_CYTOKINE} vs PBS en los cuatro tipos celulares", fontweight="bold", y=0.995)
    plt.tight_layout(); plt.show()